<!-- dd:dd-lesson-eo-1 -->

# Rearrange

*Einops · `eo-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "eo-1"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtXely20iSfhWG/4y0q2bXiaMjJmIfYv65FQ5ZZtuOdktaSW7b2zHvvqg8qrJwECAJQMf0TJsUQbCASlRmZX35ZdZfb6zz"
    "b37Z/PXmt5vm7c3D7Zc/d28uNm+urx52D82Rt3+9edg9fr17d337YRfO+PzH3e394+bx9v760+bqYfP4683V/f3mn5vH7dX9"
    "1c3H3VmxVefb+93Dp6u73Zm+2DT/mYuNPcdmv3wJrZx9fvh88/B4dXO9O4NrnjWNnF80jfxrd/NwG/5Mh+GXu+93u+vH3Yd3"
    "zR/30MS/7r/uLjZvm/+prWousFWXl+GzDp8cfzLhkw+fLi/P3/z7YjO5Q2exR3Vxsfnw+ONu98/H7W9fbq8erTnf/LxxqZtN"
    "Bx38t3g31bb0jUy3zYvZGh/6aLH74bgLxz0dL8LxAo6X4XhFx+twvMZ2VPhC6/ANiEs1f4XmVXgp4SJ0DQPNg2hLuAhdw0Dz"
    "KryUcBG6hsHmFbyWeJV4EbiGgeZVeCnpInANA83DYyvpInANA82r8FLSRfAaBltX8Np8hVd5q6FxbfDq0Lh2JAMNF9Aev4ML"
    "6JK/g4voCr+DixhF3xm4jCHZGPjORulpg/cCT0TjI2kuyVc0eC0FryVekq9o8FoKXku8JF/R4LUUvJZ4yXRFvKDBSyl4LfmK"
    "eEGDl1LwWvIV8YIGL6XgteQr4gUNXkrBK8j1QBV63N7e7B7OyAqE/84X1A99jJZHJXcts2XgP720PoebDjKPJgw6cNl04Y2t"
    "qlNN88PufwdNczDK4x1sWujrYDg81kGFXZOGObxL0xze0ThPfWrYo2SbjRuzzRYMs1m0n8Euk4FAg4YfyF4aafeilqEJL+lE"
    "j4ZVTGE6N+/4gexiNDhoSYX5tPTB40wAWnuYbHOl9efLCY662X454o77dHiaAp8+vk1rfFvuASqxPdm/+vzHx33+1QTnqmmh"
    "r5Ph8KTBPe5bTX1g2JdD1NeBmTIL95BcEniA5Fwp4fyADNABiupKqpg5TuydkSNBHpTx0scxPnlrbCZKctsOVlkeG1Jl7fly"
    "wtJysKc/Dr7f4xX21CfdnXCTqqqZl0Lt7tFqaGmfwvA4znyKo5Y+k6ZXfHBrdCtqoCKlMr4zwRpUVI+nlaR7PL1eppnHsvkC"
    "WaFaGt+ZXu02LXdQoy3p62t3ibOZxvJKXq/2nNOUY/NVPKprWZ8+s8Kh3eeb27uH1P1HuPOzm7ttM+I/nP3j5w+7L49X726+"
    "/vF+d/+wvbn78Y/zc7BlD835za8ygTTX/CKM1cP5FgTYCOLzzWOjWfe3X28+ZN//9uXqsbnm2fn28faM9Kxw55v/ahZv55v/"
    "SQ9E/qi5m92Xs/OWejY/G5LyWYEOv4eFr2p+abx1dV0YVamqai412YZnUiMxxJvUzX0b+mfT7X1tel/1rLAm+C4PAwb/YcKA"
    "ai4AituMKDD7zTDCcdVIow5/lAFCCH9UAT64PGgu65ODsJ3hAQYhBGG0BdEcK9pzwzQ/5zRp6CB1EEURMI7mooqNom2OuOZI"
    "gT5OEFVj/YsiUzpdLwwX2tXmyDiZ9JmY5YBCXI+6pTvoECkTs6UjBM3Q1Mh4YSXnSkcoXrYU9YgYipkSndU6Oro4RYYjYp5E"
    "dK2ObRkEJBESJC8sNFMwWGjSyrggbI+OlJ6hTEATsT2LsCPifip51yUjgiati8stQ5BwBEFRgiU7kGFBeBh22DBwZyQSSMhb"
    "sRVYHWONBSN4UYqW0DcED7Fdy8ActBsXDZbASDzIxwguLD2hj9SuI6gSsUFsF58zYYqGsQgNzxpxP0MLG41Pm6BOABepXU9I"
    "JGJ/SgKYNeOCRqKM9ZZxSTxG2CmhlX+Dh4s5+pmTlJlqU74O5IEttBMrt9eHOnQAhLQCkVhEB1NIllAiFB2YAU7LcYs5kQcz"
    "TUNnQx5eGOCQpr8W5ODc3zp6rI5OA/bn0lEIuvYpKgHw6REbRA1QI5OXYwg7YMVklVQxbCv10vgUaT1BO2kpy5CHiRA1rbAv"
    "F1XYODaMvODx9mVpZBNHMb9JUF8ejtpbazszAiF7h7fOyzZavFr2VF1YyOIqrrzk59jpHv3a0BJY/rqAKBH9+rgVb//dktHZ"
    "f1PYHxiLeH0SqFtDoLk8QQtzaQ7etyIYQUcYwZAkLf5yRkEGOdLsHq4Kf4VV0L77w/Wzp845PDsK168k3P0i2ifc/Y9lZuHi"
    "IB2WJd09PQMpyOIJ1L5HHjhJjZ213zS0fjtuavquMKf5ONnYjUiDPa9az+x5VV3XcjnfRMWAuRHTrItgTUHQyML+lxkHsk7s"
    "I3tbwuVSye8S7IdIvCAHjN0tlXwuQY+IzAxyvtjVUsnfik7XKbHYYjmPZYA9QQuOOdZHjhZ1Cz5g4VsZQZ9I66PydWipE7FG"
    "E+Hm16ul/drYp3Vx6djS6X7dnV1HzaKrilwpU0+P1dLmtq5/P3v7uP2/3f0tdeAc7i116Pz8cumH3e4D6qovnzwei+HYt+py"
    "X0R2JCB7TDz2mHCsS4HYwthS11qXpqiq48OP0s5ZisC6wQjs4oYhUv9hwm00telocMIw1lqHUA6EXCFwEkjMgbscRAJHQniy"
    "xBClhkCtgdhJCFOEkMIpYdpOlHYoQlu3A3dmSWnVSMxurgIygnAscjnDQg6kWDa3UYHE6vCDWs6UZubAbNUlLy3L8umnZ0WQ"
    "jGbPNGceHabVZjKpSS/aXYYTI3uJ47At7ElEAzgGkFGoTTzDpIDpKQE36v+CAbe+STEjP0/+5uBe9kyiGLu4aMccx6fSE59/"
    "HtAghoWbWZFDULtHk+3ymtwBnDtaHCkKNdMCKJj/mrRb4scxSyDXW6n7SKiIio6MiDm02kxzd2fQ6iF/d4pPfLI+m64+m5X0"
    "WUYOVF/IwLritUzT44vaF846FitbsZxNOT1MouIVr1jm9mfzZEvalMnDBCpe9Yql7uGBv7+n8qeZykUPaDVsZ0jee3f96evN"
    "7w+D+q6nqkFqayDbib89ykkX0PORLnrW12QIAh11IqFyXQmotgHIYKxMpTtZgVnmX5bd1w9R5wmCWRJgnugHtGY0wSYlThf0"
    "K0rSNvSBCJCwGkc/zNAHLzOsVWJ51ltmc4bucWq34Y+Rmxk+cja24Y+CaXnUmGilOE3gGs32wAcoAMf04aS8oPl1eJCWVNqF"
    "oLw/bv/cffj8x9lEUA/k8pPOwD33ksC9ME4bRXPGuEJXtnZz4XuDsNVqiNVbFTkcxE6AKH+F5oCmeWR4BDOFBwDZQqBLX86G"
    "4dmhdJMAgLadRruwRDQxBjDLwhYEYYFcCga0EOKkE+ErR195kGhBIq0Q+SQOAgqbuSAg7qLKKgaoV7C6yFZRplVEwImiATTf"
    "0eT2OtYbGRtY55mNWiZqtEiHViZUWIkHChpi2aUeenbf8sxHJ5IenUx3PHEdolcDH3qwhlcHFdavE/M/fTnxzAGFzspheH0w"
    "sArolglplQIZWlUMrR2euWIPcXI7n2dScdNVcbM8hKD6ABFQdafUq1B1Fw2yxA2LCB6+GjU3sbedUgWs/CZmYHbLFHQy3xOC"
    "aHwLN7Qx+7KnPMEpOXdPHCx40aqsJVNGKHJV1c8/ZwCpGy4xyS1RN2bOFOAbMzEF/XKE3U7dYWHWah1hwuhkPjglLaBIxdoW"
    "1mMjgu1pKK4H283MnZQRhQzpBCNylpkHSdz6WadnZD8kzlH62RLZA3xz+9MyujQoQUM/eQQHQGvfxO7Of7351Jxg9hu80MzF"
    "5lOf0eOvnozAQ10UgE8xYaLXhnpu1+l5iwQ7RGwdIqtnmbyTCOuZP5+FC7NIABVT6MQBqChCJwoAFSG4BsShz0hMtP6c5K8X"
    "l//plQTbWtTOoy3WHUv7lsSSK+RnrhDaji5CYO3Xm7tx+9E0dLG5G4hMhG+OrIe6x3wcXTR0Ulgx1MahrrtVuh7thwT5+lb4"
    "kVY1wYKkcif9SEKUa8eGoLOQWxF8AGUq+dyKJkY6V8eOhMZiNDHWiYlVqAdiiamyi9Zc1EVEEo+tE9pTc4hHuX2SUZ6r95EF"
    "W5nTRj3RS/ekVZGhr0qDqK14cnLAH7v7j7sPE4zV+3FjhW1dbN739T99eWwJZ1wanGi0Yn8PoTYaWKhpkoJdTQo5kznjKVOa"
    "apvcnFEgL3Ocr5/NfLjkcs2w5yQWvY5YZiFDHtPpVPQh5yCurRqZUZCFidQzZBe8qFKOwRDA7BgYBhZLOXrlnClqr+pyrlKO"
    "kEBERAO9j2jgJtWSP7F4IWInwDmglHygHiBywvXbsFwaFjfjWmYUbA+vMW2eUBYgLKBzQrwFpiZge8hmqGUU3uA9wJWMnbNg"
    "ZCgWWeyhMph8RbJO+UxD9RwK8IgjscNQslYRk5agBB1TH5q/w/2B+GwYo+FWnaw0aZvfuFhp0oWSiJEq0cybYe40nPzkm/M9"
    "Paci+KNhmsO7ar4pmvPLlreh586hKHoSoxFbuR+3qU1jF5shPDl8c1BCRXQZKyrLOFPiRDEaPalS9GTFfqu9vOsMDREYRtz9"
    "pUVrhGOe9oXpELSHWNlpXSRZkWKrlxZFMm44Y08Jw4iwHAlcLyzwNXjZv3398iUdghHcx6cYOvVyvbHXU/LJLA+BF2zVp0Lc"
    "s0HbBcdFB65MlaQsxYZmu65j32H4ugC3x4ewfJkoCxnKg9KwNCt6qm1UNsZlPnk0HS73DIL+eMI8l1aiOvHUp7FCXSm6KA48"
    "lLl5CvH03cje58T6EgfLfLcCY26P5vCYpKdUrRC7y3o8FA9LIpwxJtf30EevTyfOdxf14EXrGPEr5ig8FRYObzvAX5jsUhIk"
    "Hrtccn2wl4mV58HmWZHqMHCDe3xElu/55rfb+827ZsW+odjhsiIZIPntI+zJ0qFtCmAS40to9eAHur8w79IPapBpfkxHJlCE"
    "+1zakJy/eD8zIA4pNy1MrjgZk/tuPuzda3AEqGh+3tfHcPjEcgPTHyZ2IRmZakKZcLNwvw7JNDy0pwIqpipkS3WlPOI5SJUa"
    "3YbwdDG3X7Jk3uqls2+HtxvFr4eiVceX4RhVHgy82jW6PaJE3b00h2a4LMC+J2X3uW93wTGZl7Phxch8LXJX3Xz+9ZSKIgvO"
    "3WMlRRbznTl3dF3feeJ8N+NpkxV/ekL/JDPygvzkvWp3eDei+2vQ/fUE/raPlwu7xT7lkVL6qMwiJUtSPMM49burWOQSP7+f"
    "VPTy3dXFBs4eiWGL8w6PZYsfHxHThli24oB2Wfuy8k6Xui5OyJkHcaVFyUi2PMv4fR4Y68Ri/3vjD95JPZNP/7BN30/LNI8J"
    "+B4HMxzwPeHx8IJp+HjAnxS1JpnS5MhZ9W0hJUmydqczy/bJa4qumQia/2BWSH+WcFOl/JML4i9ytohVn1xrA41IBgnGsOFF"
    "tozCYyN2lZrrEzR/9WyAQdH1ObBBt4ZsOnBFSvYnZyFL+Y9bf6W8/x4U7z+jvWPGxd7tjFd73rPsN5r1ajm8caZOT0AdrXvO"
    "HpZueVj6hXtY4FwlymBVFc3/69Lb2qjlPSyuEzDVw9JKdX+8vouF05fCT6ond1MrJB2qnlxOrbDSj5rT4TKTHa50Zv28HC5k"
    "XMJtAbeS/xxyogbPFwEM/zrdKMKiY0guN6z93y7tPvG4ejr3KUfqE+wiXtirwKSq/DXtpNp+Fb5I9iLQHPGSXJwyEhLTq0zR"
    "yl97yP8Cm085XeQM0dlG7DgZy0PFwk7t11PdpT6IdzXPoZVWFDc6tjM7TG3vyK7mHXX3a569MsS7981vZ0mMeNxeXz2evYUG"
    "Lxv35Oyns5+Kzc8/b+AI+jvNhNm4Hm9/KS5fRg6Fy/Emb41x1pm6suVsCRSRzb83gaJgQtKiCRQtyKiTANHCi1r5EEj6z7Ir"
    "gPjfzY9o5VuEb1oZE6k6Itkv3B4ScWUsGcmJBPh7rIIIV7FwdYu1Ff2MyRfhUek9yRc2e156+eQLhWVXyN2BhAsfky3C9VGC"
    "gSgaTiW5QmpFc77DWpGBGx/SKXRMpWjOLzDKG1oP6RPkdpWUU1SCtKvmClVzflXG3WYCJRZvB/buCTfEfi25YG7JKLljqUN2"
    "o57IAH+vByng7/UhwY/WDnWtWpQYNT+9EmU56nsVaR8HEoNdQwzj6enjeed5tG0wi31aenq3xMVAcvqUrHNUgNEs9kOS09Fg"
    "xhwUTDvB+rCcHyIzRzTW+MJuaqq8qT2eCv3URZJhyi0JaVh0E1gdRIPsdNxaD2p/aJSLpjiXxvIfGnqsy1m2FYG8cxiPevHx"
    "uEIBwKEan6uanr6an6Ze3shqzPD6OLWnH4d7+lEfmdzWt5ftsmbVxtS2NTuu2huDZBX8RaKb2Bgkq+cjs9Eye5MsSjs1LSXA"
    "oa2B155sOFF0KDOpwoYLS51VExLWNbN1uTXrGNTMsCXT1aIdZ7uAZrsXiF1Y4lzRtuBpbwI0nvBK9lDBa2fXgsyszmw4Ibfv"
    "41TDedqIe2YbsqypaT17MVXm7zoEL2cNbSasoc1qa2iJ8pu4ywHXboyl+2vaEQDXzRQToDV1WkWz5ZVrZkUrZ/xGrrNT5YG5"
    "1752vPBAwfuvLL327dQd4G1i+6oOBIEM1hyIy+K+igPh01C9AdiwYajaQFwyN+eX+Nybb6owF0E7sHgON1/FBXRzdg3t1M3x"
    "GnpFYSVYS4eOqSIr71YthghO31N5CP/TZgAAbIb7y7BeAQAJtsvBJsxaaW+qwlW+MWDGVTOZLx13atm7W4s2yQFdVrE0gUnh"
    "akFnRI1f3H6Z9ChuQmxLBJFcHeTV/EOIrkKgCTRHk5bETYpLS/BRcDrDaEf2k0OICfSiRB2oCWgEdQCgC1zYcJcWbhPhRtwr"
    "Gu49HDXhbm24XetlGVbeNtoE21Bz8VfeRpqgMjTg4cGHzJKmOR9GQthk2Ufr0BwBS+AJJMP7D42FtNqg6QCCgY+rWfvLEJQM"
    "mq4jQEZ3H/6GDa6DyDX6xOFWbYnfh6ManPJwNIB6OghYWwLncMNs3gIbLBu2DGLhTbGDvWv+YS+CffNky0q0WwVIxIWNpctg"
    "4Zp/dbBmzT/oA1gvTZbKRpuEBTRCScQAATaHavDYw50quv/wN2zXHQSvTZCNA8capzkQf5C/trAQCPJxYMznnMZA1TTjuPsx"
    "XD152605QFzc7rwHwg16FgFc1jeAb/Hp9YG3Qc8SdCvcWf2y6Mr4+br1+cNhdGZ8u8a3D1OpN+1fnUDEaTd1JC0nOtGNkatD"
    "CS8XtguzxSmkkd47lDR8E+n2f3a0JYT2/wyh/bOULz+BkT8klxESSevs0XVsHssyYrexzlEbvXMn9irrHr1cSNRTqOca2BSB"
    "89QMuT+T7CfSKlaTuygCwzOPFqXiqd66jq4ErY9o2kJHHafgMInRt5kjYtMkJkHPpVbqE93g72SUfjGNlfoUZh3z6823hw52"
    "Ia3U94vNp4eLzbe9/q446UAbJH55uMmpQJ8BtPNABdSBBeiLQhl3LBXwe+b1ksMbqgcOj3y0LI6qa0NvgkgvxkJaqe99Kb/x"
    "yynl7WxcV3pRuy75vnUkW9F6FAY379zn43qzhLUl01VYEYISEBwAPnXNnrOLJQYLorShT0C6FP0B9AP43DIuhOtUbA90CY2Y"
    "xSJ6eDK4XAglBN+QtxDM/MPyhPKF3/Otd0T4vM/18r3pzvlT16s99ShRkE0qH6lwHFCVQkVmSGSEnexi/RjMonUp8TcXywg8"
    "+mOfWH4cIJaB/PCpw+PHKcUEn6q/ThYYNKmGIEZSYiVBEYxxsvRgmWoLYlAlVhjMoileFhgUUWmMi8RKgiK44mXpwTIFcDBA"
    "EisMpiSLtI+WoSiSkXEYJ6MxTgZLihQxKfIYt5OBGZenv5cpXiXCJSpGnTSHoESWfJkCWGWKxaQQlOZ41GHcxR+DBRPjoNKr"
    "Dap+SmfvzhQHdnBk+/e8t/opTcbIjg1gSb2de7EacbcYgmqvccAMnf1+jmub35N/XWBNyVuQnt2LXV5swmnJo+t88/aXi80v"
    "gbLeeHFfPj88ng36YAV4PuHGJKfZihB4/9E0jlw7p6hzNJGVfXvPvfzo5UxopxuYz0i4Zi3hpqeOwQQQEVQtnA3W7eEDPlFP"
    "C9FTzWXtKvIr0+5twZiDTziXDEzPJvZCBHotEej8YWu5cDw5ufbu6vH60+5hbzFoE5/+p8ZwTvJiqNnG1ja/6DPD+QmHERXb"
    "m2QL4mKLqAjYMn6uqQk6WUdbrlVkuTHRTDFl7YBpLAnycLojbdCSpGvXlu6UiiNTCgPxfFjED31bcx1UmKR/P7B8f1+5SU9W"
    "KDurjd3apiejDbVIPTxUhHss62rTAOonONLH6OwhTbFFcUy0oBZpp0XNoUtFf5gGqHBzhxmRxIXMSD9HUXukkZAbI1viRdLA"
    "HffMZh64OjMLfXzJkcNHSeBQElCSzpMZTRl0P3nK+D48WWieod20XdW+D+zR9H36XlNGbBag82xBGy0/bx4QffcizQZEMYbv"
    "PPvyScEPzGHPYSPt3LRaVAQSriMyKkWghFFHNmSeU1h6aftNXN22yO5kuZOtJhsdzylFbiCb9NKnbR1bdHcfnwBZai8fLNPh"
    "Sy8p8CahBi3CO80EcuHPtjaehhxKk9PgiQBvBijvRPvMTD+xP+OZRtBQxTxheGboI72zbZe2nE14OrP0idUpDD+RN40w8bTP"
    "C7SAQA45QgYpyfgIYwKiJkyIgB6eJg1Vs6CHKXI84fEY1ETOI8XpG0aWsdt0ZomnGnwr+eQSzzb4FtmwIAkDI8rwyg4kYeDB"
    "miq6fygIg6PK8BRsUBAGn6+p0rZb8KQMokmUWGFANgZGnKm36cwSTzX4VvLJJZ5t8C1iR7iahWFmWcFQw9BfijpmiRdN3Oeo"
    "ZqRn+IRt0jSLW6EjCkHKZkE2FkafjfpmaZd0HH6WVc6idCwOP3soCpXZ/Lpr8m3aTdIubL9E2gA7/yl3lvOW2JpHy8K6LhYF"
    "B3e/j9k8YSvC03p8MKc5TvVOPUvGxKdv19M4EM2JY4QHOOUIdgP87vC4ovFKseMbyAxFXdXOVYUuCndKiREUSdr+mTbK0vuK"
    "jRTT6oWEng4EycNXh29L1kmsbW9Kxlvdt1jEfQm37a3LLk+WYU76tQPROtNJjFlDkoTTOcp67Ww3hsIIk28Ma2YbjuHUEyYN"
    "zCHubjkWiX8mhkCzTcewfl6g/sFfnW3HYMUe7g+u1d14zKk5is/uXT5MXzs0LQ0YVvrmIHxflFNPASfbQfgx9Ewbk4kVhNgY"
    "VWxWlhbr/uCy+wcCSQetIWYSnerfPiEDgBIs004ZbSMbaSdVWV5EyYoiGZqUMB5uOYNnBEgi68no/qLXWf5XCzBqIz0Cb2Ht"
    "k8hYwqu6cJOEfcgE5rCRzAnjtgXQluAvaluiVwJD4rYzDEogQXIPX9dB0yIsKsGwTvJvK88tA5YOHOgivmlpI9sJnuTpo3hk"
    "e9qj9rQ9ZZPblnHsLVEzyeWcRcEHC9zxmM+4I87OnLjrujMDBWDumvc7MykOARl0eP5ggh19vTiDpJ21WxcTsKEqFkN42l7H"
    "+lKa6j/59nYknhkTLuXzeo70JgyIMqFFhcxSVFPwKTHXbz1n57YwonLruS5CmRKGK4n2KbSMJlE86q2oaFAIzKckhAjtf8KA"
    "6gwC4kndJ56J3P87ugE+JShbmY3sMqyo2PpUDaGQicpldCskxmMZFUppxhH28RnsUzBWRLY52mRCdxSgOgjxGIR4Ss/YkFEZ"
    "NoRzJD0mwmk0YztI6pOYjq4ZBSLkSMI8OoN5DGNDBCZF5GeedOOoKnpVVVmCHbOvmkHGj5F91k9tFKdwZKx/jvukT4ImRmCJ"
    "YyCJo0vKx4KnAYywlS/LytUn5FVkUzBRnIeBCEcsn1F3ZMAXmZaS7ERZr5QK4WOOg0xJJjhCyT3Sa4FQaE5JvjxJRjk1uBjP"
    "LEYii11QUimpOE8HztOK83TgPK1YpAPjbCuSk/Ok4jw5OU8qFsnJ7S3ME3FFzxqFbJvFKjGNm/dvoQqDu5gQWcOzhwjY+OVx"
    "JlFCDF2DeERo8UDvMcrBrCsHK4vAGPYc7TZRRepU5armWlexlhXXhonluJws32JEUaw6lcZCZ61kxy2LVjpZ26X0iSgYwQr0"
    "HUuOJpYcRNRa1vPyqaqXl0VZGJswyXestrF4VsU+Zcdz9HlpG5M8Ry6WVVG8TkXfjVwz8t10Xn/LyzJcRV6gxnjpvWmO9Sn2"
    "zNh7M+3oZJHXrylTxJL91pp8uuRasueoyOfLqnmVsqhXmdegIc9RsX9IDqWPnqNin/KYWFaf3xiVRK+mJEv4jN+nLaOfyiT0"
    "L6FxXqjn3mw7kwVk8YSXKvzL6Z1oLSuQS5gufr35pkcT4LDbIwlwdNIRCXD0yyMCVViWIEzZwb2tTF2r2qv5k9/cePKbe6Kh"
    "RuEWGWZJwRUMqOBRypKrkTwMhwKZGOnFwCCIJRiK5E+J3E8jIjMYeKbtcCCZjrkEGtLUoNIrFGlYIjWt2F8VIOaPFC1jZ1d7"
    "JikZsIyJteiyQ+UHFR1PDA6GBxVqv6Yol/PscRac9BcKCsBjhQICHPEyXFoAvNo6NzPVGrxpkQb4dLxp6Tf53PKK2oMcT6lT"
    "dlikN8V4Vpzs12NGs/f61Mzo6LfKIrCimGGs0NqqHJs4c7HU4GgN13ahQ3YDuwVUO5VeJd9MeHxDdVazWrVZsUMfcdasYmyr"
    "lm3i8IlahOM1ZduFEJ3ARFuVXDuVZyXvLfMWj6/4OsRujl7h07GbD2L+zFDn8IXSnEU3iacwdzCqj+ZcoW3/5oK7OAFx/eYG"
    "0dbmq4N858hASIVDO9a6i8JKroIIppzAU2iHtExgn0zhOwcQam3ZCTserXQWVM8MWSHzg5J57a2P3bbNmcllR5GCI2IVq3j9"
    "jbYVWjEELHjivRL/VOWbpOR220QgxMjquXKSwV/3WdMWIaNgjCA3xpE4gK4sww8ZJ1gxYgB7HjBllomzwMRF9m5sJ80j3erq"
    "+eQjJpYO3aM921EDYjIAjCThJjq2QiGuyEFWkonMLOKMRKyISgxvMWW8NcVnKUsVwVP5FF9LeknZKdXbyumpBERD4AxF8VI7"
    "BtuJvGniRDMzOicuK0lfTiRmrgxBNGaiKGdEZQz5MumYyMRMKW7e4lh1WIGAqg9gcQEqLMCcK2jFUcwX473YiouteGjFYyse"
    "WvHYimf1Rf0lBUYNxlZ8bMWSL6aJmV1KMjVRr3HEWhixFkesxToAacQ6Qh4RSiQYEX0nbsdhOw7acdiOg3ZcasdTtQQshkCF"
    "EBCJ5HY8tuOhHY/teGjHp3as2TLJG1jjyDQnCjl2C8sS4Pi1MH4tjl8bx6+DVhy24jCcT6F8asVBKw5bcdCKw1ZcbMUjGEp+"
    "Ie7/iK14asVDKx5b8dCKx1Z80qVIfVdEgIc3fCWmAI5iC6PY4ii2MIpt0gZHSolaSWqJesntOGzHQTsO23HQjkvteOIl4DxK"
    "dARMc4k5idiOh3Y8tuOhHV/PE9SGJDuYK83yc6XuUOzzP46OXbve2LXmrulV3IDhzd7Re/z3/wNwR/A9"
)
print("Delta Drills checker ready — 62 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-einops-pattern-language -->

## The einops pattern language — naming and permuting axes

`einops.pattern-language`


`einops.rearrange(tensor, 'PATTERN')` is reshape/transpose with the axes
spelled out in words. The pattern is two axis lists around an arrow:

> `'b h w c -> b c h w'`
> — left side: name each input axis, in order. Right side: the same names,
> in the output's order.

einops names are whole
space-separated WORDS (`batch`, `h`, `nh`), and — the big semantic
difference — **every name on the left must appear on the right** (rearrange
never sums; reducing is a different function, later KP). What rearrange
does is exactly what the name-shuffle says:

- `'b h w c -> b c h w'` — channels-last to channels-first: axis `c` moves
  to position 1, values untouched.
- `'b t d -> t b d'` — batch-first to time-first.
- `'h w -> w h'` — a 2-D transpose.

Why this beats `x.permute(0, 3, 1, 2)`: the pattern is
self-verifying documentation. It states what each axis MEANS, the library
checks that the input really has 4 axes, and six months later the intent is
still legible. In deep-learning code, layout bugs (bhwc vs bchw) are among
the most common and least visible — naming the axes at every hop is the
antidote, which is why the ARENA curriculum drills einops before touching
models.

Reading discipline: identify each name's position
on the left (what it is) and on the right (where it goes). If a name
appears exactly once per side, the operation is a pure permutation — data
moves, nothing merges, splits, or disappears. Merging and splitting add
parentheses to this grammar — next two KPs.


Task: channels-last batch → channels-first, and batch-first sequence →
time-first — with one-element verification.


In [ ]:
import torch as t
import einops

arr = t.arange(24).reshape(2, 2, 2, 3)      # (b, h, w, c) channels-LAST

# Name the four axes; emit them with c pulled to the front block.
first = einops.rearrange(arr, 'b h w c -> b c h w')
assert first.shape == (2, 3, 2, 2)
# Track one element: input (b=1, h=0, w=1, c=2) must land at (1, 2, 0, 1).
assert arr[1, 0, 1, 2] == first[1, 2, 0, 1]

print("channels-last", tuple(arr.shape), "-> channels-first", tuple(first.shape))




The image example moved the channel axis; the same grammar moves the time axis of a sequence in front of its batch axis.


In [ ]:
import torch as t
import einops

# Sequence layout swap: batch-first -> time-first.
seq = t.arange(12).reshape(2, 3, 2)         # (b, t, d)
tfirst = einops.rearrange(seq, 'b t d -> t b d')
assert tfirst.shape == (3, 2, 2)
assert seq[1, 2, 0] == tfirst[2, 1, 0]

print("batch-first", tuple(seq.shape), "-> time-first", tuple(tfirst.shape))




Finally, the pattern is an EXPECTATION about the input, and einops enforces it.


In [ ]:
import torch as t
import einops

# The pattern is checked against reality: wrong axis count = loud error.
try:
    einops.rearrange(seq, 'b h w c -> b c h w')   # 3-D data, 4-name pattern
    raised = False
except Exception as err:
    raised = True
    print("4-name pattern on 3-D data ->", type(err).__name__)
assert raised

print("element (1,0,1,2) moved to (1,2,0,1):",
      arr[1, 0, 1, 2].item(), "==", first[1, 2, 0, 1].item())




Why each step:

1. The tracked element (`arr[1,0,1,2] == first[1,2,0,1]`) is the same
   verification that works for every relayout — indices permute exactly as
   the names did. One element is a spot check of the mapping; a full
   `t.equal` against `arr.permute(0, 3, 1, 2)` is the proof.
2. Note what the names buy in the sequence example: 'b t d -> t b d' READS
   as "time first"; the transpose-tuple spelling `(1, 0, 2)` says the same
   thing to the machine and nothing to the reader.
3. The deliberate error shows einops as a shape CHECKER: patterns carry
   expectations, and mismatches fail at the call — not three functions
   later. This is a feature to lean on, not an annoyance.


<!-- dd:dd-q345 -->

### Problem 345 · faded — your turn

Channels-last batch to channels-first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 3.]],

         [[1., 4.]],

         [[2., 5.]]]])
```


In [ ]:
import torch as t
import einops

def solve(arr):
    """(b, h, w, c) -> (b, c, h, w)."""
    return einops._____(arr, '_____')


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 1, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(345)


In [ ]:
#@title 💡 Solution — Problem 345
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b h w c -> b c h w')


print(solve(t.arange(6.0).reshape(1, 1, 2, 3)))


<!-- dd:dd-q388 -->

### Problem 388 · guided

Write a function solve(seq) that takes a batch-first sequence tensor (b, t, d) and returns the TIME-FIRST layout (t, b, d) — batch and time axes exchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.],
         [3.]],

        [[1.],
         [4.]],

        [[2.],
         [5.]]])
```


<details>
<summary>Hints</summary>

1. (b, t, d) to time-first (t, b, d) — name the three axes, reorder two of
   them.
2. All names appear on both sides — a pure permutation.
3. `'b t d -> t b d'`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq):
    """Return the TIME-FIRST layout (t, b, d) — batch and time axes exchanged."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(2, 3, 1)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(388)


In [ ]:
#@title 💡 Solution — Problem 388
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq):
    return einops.rearrange(seq, 'b t d -> t b d')


print(solve(t.arange(6.0).reshape(2, 3, 1)))


<!-- dd:dd-q335 -->

### Problem 335 · independent

Write a function solve(img) that takes a channels-last image (h, w, c) and returns the channels-FIRST version (c, h, w).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 3.]],

        [[1., 4.]],

        [[2., 5.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return the channels-FIRST version (c, h, w)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(335)


In [ ]:
#@title 💡 Solution — Problem 335
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'h w c -> c h w')


print(solve(t.arange(6.0).reshape(1, 2, 3)))


<!-- dd:dd-q330 -->

### Problem 330 · independent

Write a function solve(arr) that takes a batch of shape (b, c, h, w) and returns shape (c, h, w, b) — the batch axis moved to the END, everything else in order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 2.],
          [1., 3.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, w, b) — the batch axis moved to the END, everything else in order."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(2, 1, 1, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(330)


In [ ]:
#@title 💡 Solution — Problem 330
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> c h w b')


print(solve(t.arange(4.0).reshape(2, 1, 1, 2)))


<!-- dd:dd-q379 -->

### Problem 379 · independent

Write solve(imgs) for a (B, C, H, W) batch: swap the HEIGHT and WIDTH dimensions of every image (spatial transpose) — output (B, C, W, H). Pattern: 'b c h w -> b c w h'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          ...,
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255]],

         [[205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          ...,
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205]],

         [[155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          ...,
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155]]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(379)


In [ ]:
#@title 💡 Solution — Problem 379
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
imgs = arr

def solve(imgs):
    return einops.rearrange(imgs, 'b c h w -> b c w h')

print(solve(imgs))


<!-- dd:dd-q319 -->

### Problem 319 · independent

Write a function solve(arr) that takes a channels-FIRST batch of shape (b, c, h, w) and returns the channels-LAST layout (b, h, w, c) — same values, the channel axis moved to the end.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 2., 4.],
          [1., 3., 5.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the channels-LAST layout (b, h, w, c) — same values, the channel axis moved to the end."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 3, 1, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(319)


In [ ]:
#@title 💡 Solution — Problem 319
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> b h w c')


print(solve(t.arange(6.0).reshape(1, 3, 1, 2)))


<!-- dd:dd-q327 -->

### Problem 327 · independent

Write a function solve(img) that takes a channels-last image of shape (h, w, c) and returns shape (h, c, w) — the channel and width axes exchanged, so color ends up between height and width.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 3.],
         [1., 4.],
         [2., 5.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return shape (h, c, w) — the channel and width axes exchanged, so color ends up between height and width."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(327)


In [ ]:
#@title 💡 Solution — Problem 327
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'h w c -> h c w')


print(solve(t.arange(6.0).reshape(1, 2, 3)))


<!-- dd:dd-q344 -->

### Problem 344 · independent

Write a function solve(img) that takes a channels-first image (c, h, w) and returns shape (c, w, h) — height and width TRANSPOSED within each channel, reflecting the image across its main diagonal.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 3.],
         [1., 4.],
         [2., 5.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return shape (c, w, h) — height and width TRANSPOSED within each channel, reflecting the image across its main"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(344)


In [ ]:
#@title 💡 Solution — Problem 344
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'c h w -> c w h')


print(solve(t.arange(6.0).reshape(1, 2, 3)))


<!-- dd:dd-q913 -->

### Problem 913 · independent

Given a batch of channels-first images `imgs` as a nested list of shape `(b, c, h, w)`, transpose every image (swap its height and width) and return the `(b, c, w, h)` nested list.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[[0, 2], [1, 3]]], [[[4, 6], [5, 7]]]]
```


In [ ]:
import torch as t
import einops


def solve(imgs):
    """Transpose every image."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[[0, 1], [2, 3]]], [[[4, 5], [6, 7]]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(913)


In [ ]:
#@title 💡 Solution — Problem 913
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(imgs):
    """Transpose every image."""
    return einops.rearrange(t.tensor(imgs), 'b c h w -> b c w h').tolist()


example = ([[[[0, 1], [2, 3]]], [[[4, 5], [6, 7]]]],)
print(solve(*example))


<!-- dd:dd-q914 -->

### Problem 914 · independent

Given one channels-first image `img` as a nested list of shape `(c, h, w)`, return the same image channels-last as a nested list of shape `(h, w, c)`.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[0, 4], [1, 5]], [[2, 6], [3, 7]]]
```


In [ ]:
import torch as t
import einops


def solve(img):
    """Channels first to last."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[0, 1], [2, 3]], [[4, 5], [6, 7]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(914)


In [ ]:
#@title 💡 Solution — Problem 914
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(img):
    """Channels first to last."""
    return einops.rearrange(t.tensor(img), 'c h w -> h w c').tolist()


example = ([[[0, 1], [2, 3]], [[4, 5], [6, 7]]],)
print(solve(*example))


<!-- dd:dd-q915 -->

### Problem 915 · independent

Given one channels-last image `img` as a nested list of shape `(h, w, c)`, return the same image channels-first as a nested list of shape `(c, h, w)`.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[0, 1], [2, 3]], [[4, 5], [6, 7]]]
```


In [ ]:
import torch as t
import einops


def solve(img):
    """Channels last to first."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[0, 4], [1, 5]], [[2, 6], [3, 7]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(915)


In [ ]:
#@title 💡 Solution — Problem 915
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(img):
    """Channels last to first."""
    return einops.rearrange(t.tensor(img), 'h w c -> c h w').tolist()


example = ([[[0, 4], [1, 5]], [[2, 6], [3, 7]]],)
print(solve(*example))


<!-- dd:dd-q916 -->

### Problem 916 · independent

Given a batch of channels-first images `imgs` as a nested list of shape `(b, c, h, w)`, swap the batch and channel axes so each channel holds the whole batch. Return the `(c, b, h, w)` nested list.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[[0, 1], [2, 3]], [[0, 1], [2, 3]]], [[[4, 5], [6, 7]], [[4, 5], [6, 7]]]]
```


In [ ]:
import torch as t
import einops


def solve(imgs):
    """Batch and channel swapped."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[[0, 1], [2, 3]], [[4, 5], [6, 7]]], [[[0, 1], [2, 3]], [[4, 5], [6, 7]]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(916)


In [ ]:
#@title 💡 Solution — Problem 916
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(imgs):
    """Batch and channel swapped."""
    return einops.rearrange(t.tensor(imgs), 'b c h w -> c b h w').tolist()


example = ([[[[0, 1], [2, 3]], [[4, 5], [6, 7]]], [[[0, 1], [2, 3]], [[4, 5], [6, 7]]]],)
print(solve(*example))


#### Common mistakes

- **"einops names are single characters."** — They're
  space-separated words: `'batch height width channels -> ...'` is legal and
  sometimes clearest. The space is the separator; 'bhwc' would be ONE axis
  named bhwc.
- **"rearrange can drop an axis I don't need."** — Every input name must
  appear in the output; rearrange is lossless by design. Dropping = summing
  or selecting, which are reduce (later KP) or plain indexing.
- **"It's just transpose with extra steps."** — It's transpose PLUS shape
  verification PLUS documentation. The pattern fails loudly when the input
  doesn't match the declared layout — the check you didn't know you needed
  until a bhwc/bchw bug eats an afternoon.


<!-- dd:dd-kp-einops-merge-axes -->

## Merging axes with (parentheses)

`einops.merge-axes`


Parentheses on the **output** side of a pattern MERGE axes into one:

> `'c h w -> c (h w)'`
> — h and w fuse into a single axis of length h·w.

Two things define what you get:

1. **Order inside the parens = nesting order.** The LEFT name varies
   slowest, the rightmost fastest (exactly row-major reshape). `(h w)`
   walks: h=0 with all w's, then h=1 with all w's… — each row in full, row
   by row. `(w h)` would walk columns instead. When a task says
   "row-major", "reading order", or "all of X's block before the next X",
   it is dictating the paren order.
2. **Which axes you merge — and they need not be adjacent in the input.**
   `'b h w c -> h (b w) c'` merges batch INTO width: because b is the slow
   (left) name, image 0's columns come first, then image 1's — the images
   laid side by side. Merging non-adjacent axes quietly includes the
   transpose that brings them together; the pattern spells the whole move.

The classic instances:

- Flatten spatial: `'c h w -> c (h w)'` — per-channel row-major flattening.
- Stack a batch vertically: `'b h w c -> (b h) w c'` — batch into height,
  image after image.
- Side-by-side concatenation: `'b h w c -> h (b w) c'`.
- Channel unroll: `'c h w -> (c h) w'` — all of channel 0's rows, then
  channel 1's… (c slow, h fast).

In raw PyTorch each of these is a permute+reshape pair you must derive;
in einops the pattern is the derivation.


Task: flatten an image's spatial axes per channel; lay a batch out side by
side.


In [ ]:
import torch as t
import einops

img = t.arange(12).reshape(3, 2, 2)      # (c, h, w)

# Merge h and w, h varying slowest: each channel flattens in reading order.
flat = einops.rearrange(img, 'c h w -> c (h w)')
assert flat.shape == (3, 4)
assert flat[0].tolist() == [0, 1, 2, 3]   # row 0 then row 1 of channel 0

print("'(h w)' reads across rows:", flat[0])




Same merge, other order inside the parens: the walk changes, the length does not.


In [ ]:
import torch as t
import einops

# Paren order matters: (w h) reads DOWN the columns instead.
flat_cols = einops.rearrange(img, 'c h w -> c (w h)')
assert flat_cols[0].tolist() == [0, 2, 1, 3]

print("'(w h)' reads down columns:", flat_cols[0])




To put whole images side by side, the batch index must vary SLOWER than the column index — so `b` goes on the left inside the parens, even though `b` and `w` are not adjacent in the input.


In [ ]:
import torch as t
import einops

# Merge NON-adjacent axes: batch into width -> images side by side.
batch = t.arange(16).reshape(2, 2, 2, 2)  # (b, h, w, c)
wide = einops.rearrange(batch, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 2)
# Row 0: image 0's two columns, THEN image 1's two columns (b is slow).
assert wide[0, :, 0].tolist() == [0, 2, 8, 10]
print("batch merged into width", tuple(wide.shape), "-> row 0:",
      wide[0, :, 0])




Why each step:

1. The `(h w)` vs `(w h)` pair on the same data is the fastest way to burn
   in "left = slow": identical merge, different walk, different output.
   When unsure, test both on arange data — the values ARE their original
   positions.
2. For the side-by-side merge, predict before running: b slow means all of
   image 0's width before image 1's — that's "side by side, image 0 on the
   left". If you wanted interleaved columns, b would go FAST: `(w b)`.
3. Notice `wide`'s shape (2, 4, 2) contains the arithmetic (b·w = 4) — a
   merged axis's length is always the product, a free sanity check.


<!-- dd:dd-q391 -->

### Problem 391 · faded — your turn

Flatten spatial axes per channel, reading order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
```


In [ ]:
import torch as t
import einops

def solve(img):
    """(c, h, w) -> (c, h*w), rows before columns."""
    return einops.rearrange(img, '_____')


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(391)


In [ ]:
#@title 💡 Solution — Problem 391
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'c h w -> c (h w)')


print(solve(t.arange(8.0).reshape(2, 2, 2)))


Task: flatten an image's spatial axes per channel; lay a batch out side by
side.


In [ ]:
import torch as t
import einops

img = t.arange(12).reshape(3, 2, 2)      # (c, h, w)

# Merge h and w, h varying slowest: each channel flattens in reading order.
flat = einops.rearrange(img, 'c h w -> c (h w)')
assert flat.shape == (3, 4)
assert flat[0].tolist() == [0, 1, 2, 3]   # row 0 then row 1 of channel 0

print("'(h w)' reads across rows:", flat[0])




Same merge, other order inside the parens: the walk changes, the length does not.


In [ ]:
import torch as t
import einops

# Paren order matters: (w h) reads DOWN the columns instead.
flat_cols = einops.rearrange(img, 'c h w -> c (w h)')
assert flat_cols[0].tolist() == [0, 2, 1, 3]

print("'(w h)' reads down columns:", flat_cols[0])




To put whole images side by side, the batch index must vary SLOWER than the column index — so `b` goes on the left inside the parens, even though `b` and `w` are not adjacent in the input.


In [ ]:
import torch as t
import einops

# Merge NON-adjacent axes: batch into width -> images side by side.
batch = t.arange(16).reshape(2, 2, 2, 2)  # (b, h, w, c)
wide = einops.rearrange(batch, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 2)
# Row 0: image 0's two columns, THEN image 1's two columns (b is slow).
assert wide[0, :, 0].tolist() == [0, 2, 8, 10]
print("batch merged into width", tuple(wide.shape), "-> row 0:",
      wide[0, :, 0])




Why each step:

1. The `(h w)` vs `(w h)` pair on the same data is the fastest way to burn
   in "left = slow": identical merge, different walk, different output.
   When unsure, test both on arange data — the values ARE their original
   positions.
2. For the side-by-side merge, predict before running: b slow means all of
   image 0's width before image 1's — that's "side by side, image 0 on the
   left". If you wanted interleaved columns, b would go FAST: `(w b)`.
3. Notice `wide`'s shape (2, 4, 2) contains the arithmetic (b·w = 4) — a
   merged axis's length is always the product, a free sanity check.


<!-- dd:dd-q347 -->

### Problem 347 · faded — your turn

Same input, a different merge: lay the channels out HORIZONTALLY, so channel
0's whole image sits left of channel 1's. Shape (c, h, w) -> (h, c·w). Two
decisions the flatten above did not ask for — WHICH pair of axes merges (they
are not adjacent in the input), and which of them is the slow one.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1., 4., 5.],
        [2., 3., 6., 7.]])
```


In [ ]:
import torch as t
import einops

def solve(img):
    """(c, h, w) -> (h, c*w): channel 0's image, then channel 1's, side by side."""
    return einops.rearrange(img, '_____')


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(347)


In [ ]:
#@title 💡 Solution — Problem 347
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'c h w -> h (c w)')


print(solve(t.arange(8.0).reshape(2, 2, 2)))


<!-- dd:dd-q357 -->

### Problem 357 · guided

Write solve(img) for a (C, H, W) channels-first image: combine channels and height into one tall strip — output ((C·H), W), channel 0's rows first, then channel 1's, … Pattern: 'c h w -> (c h) w'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[255, 255, 255,  ..., 255, 255, 255],
        [255, 255, 255,  ..., 255, 255, 255],
        [255, 255, 255,  ..., 255, 255, 255],
        ...,
        [155, 155, 155,  ..., 155, 155, 155],
        [155, 155, 155,  ..., 155, 155, 155],
        [155, 155, 155,  ..., 155, 155, 155]], dtype=torch.uint8)
```


<details>
<summary>Hints</summary>

1. (C, H, W) → ((C·H), W): channels and height merge into one tall strip,
   channel 0's rows first.
2. "Channel 0's rows first, then channel 1's" tells you which name is slow
   inside the parens.
3. `'c h w -> (c h) w'`.

</details>


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(357)


In [ ]:
#@title 💡 Solution — Problem 357
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    return einops.rearrange(img, 'c h w -> (c h) w')

print(solve(img))


<!-- dd:dd-q342 -->

### Problem 342 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and MERGES batch into height: return shape (b*h, w, c), all images stacked top-to-bottom in batch order — 'b h w c -> (b h) w c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.]],

        [[4., 5.],
         [6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b*h, w, c), all images stacked top-to-bottom in batch order — 'b h w c -> (b h) w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(342)


In [ ]:
#@title 💡 Solution — Problem 342
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b h w c -> (b h) w c')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q314 -->

### Problem 314 · independent

Write a function solve(arr) that takes a channels-last batch of shape (b, h, w, c) and concatenates the images SIDE BY SIDE: return shape (h, b*w, c) where image n occupies columns n*w through (n+1)*w - 1 — the einops pattern 'b h w c -> h (b w) c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (h, b*w, c) where image n occupies columns n*w through (n+1)*w - 1 — the einops pattern 'b h w c """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(314)


In [ ]:
#@title 💡 Solution — Problem 314
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b h w c -> h (b w) c')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q346 -->

### Problem 346 · independent

Write a function solve(arr) that takes a channels-FIRST batch (b, c, h, w) and concatenates the images side by side WITHIN the channels-first layout: return shape (c, h, b*w) via 'b c h w -> c h (b w)'. (A companion drill does this for channels-last input.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1., 4., 5.],
         [2., 3., 6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, b*w) via 'b c h w -> c h (b w)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(346)


In [ ]:
#@title 💡 Solution — Problem 346
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> c h (b w)')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q353 -->

### Problem 353 · independent

Write a function solve(seq_chunks) that takes a chunked sequence tensor of shape (b, n, p, d) — n chunks of p tokens each — and MERGES the chunk axes back into one sequence: return shape (b, n*p, d) via 'b n p d -> b (n p) d'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq_chunks):
    """Return shape (b, n*p, d) via 'b n p d -> b (n p) d'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(353)


In [ ]:
#@title 💡 Solution — Problem 353
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq_chunks):
    return einops.rearrange(seq_chunks, 'b n p d -> b (n p) d')


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q373 -->

### Problem 373 · independent

Write solve(img) for an (H, W, C) image: unroll the CHANNEL dimension along the height axis — all of channel 0's rows, then channel 1's, then channel 2's — with a trailing singleton axis so the result is ((C·H), W, 1). Pattern: 'h w c -> (c h) w ()'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[155],
         [155],
         [155],
         ...,
         [155],
         [155],
         [155]],

        [[155],
         [155],
         [155],
         ...,
         [155],
         [155],
         [155]],

        [[155],
         [155],
         [155],
         ...,
         [155],
         [155],
         [155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[4]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(373)


In [ ]:
#@title 💡 Solution — Problem 373
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[4]

def solve(img):
    return einops.rearrange(img, 'h w c -> (c h) w ()')

print(solve(img))


<!-- dd:dd-q380 -->

### Problem 380 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and returns shape (b*h, w, c): batch merged into height AND the layout converted to channels-last in the same pattern — 'b c h w -> (b h) w c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.],
         [1.]],

        [[2.],
         [3.]],

        [[4.],
         [5.]],

        [[6.],
         [7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b*h, w, c)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(380)


In [ ]:
#@title 💡 Solution — Problem 380
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> (b h) w c')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q392 -->

### Problem 392 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and stacks the images VERTICALLY in channels-first layout: return shape (c, b*h, w) via 'b c h w -> c (b h) w'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.],
         [4., 5.],
         [6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, b*h, w) via 'b c h w -> c (b h) w'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(392)


In [ ]:
#@title 💡 Solution — Problem 392
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> c (b h) w')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q400 -->

### Problem 400 · independent

Write a function solve(arr) that takes a batch (b, c, h, w) and merges batch into width with the batch index INNERMOST: return shape (c, h, w*b) via 'b c h w -> c h (w b)' — the images' columns INTERLEAVE (column 0 of every image, then column 1, ...) instead of sitting side by side. (Contrast '(b w)', which concatenates whole images.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 4., 1., 5.],
         [2., 6., 3., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, w*b) via 'b c h w -> c h (w b)' — the images' columns INTERLEAVE (column 0 of every image,"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(400)


In [ ]:
#@title 💡 Solution — Problem 400
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> c h (w b)')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q889 -->

### Problem 889 · independent

Given a batch of channels-first images `imgs` as a nested list of shape `(b, c, h, w)`, lay the images side by side in batch order (image 0 leftmost) and return one image as a nested list of shape `(c, h, b·w)`.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[0, 1, 4, 5], [2, 3, 6, 7]]]
```


In [ ]:
import torch as t
import einops


def solve(imgs):
    """Batch side by side."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[[0, 1], [2, 3]]], [[[4, 5], [6, 7]]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(889)


In [ ]:
#@title 💡 Solution — Problem 889
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(imgs):
    """Batch side by side."""
    return einops.rearrange(t.tensor(imgs), 'b c h w -> c h (b w)').tolist()


example = ([[[[0, 1], [2, 3]]], [[[4, 5], [6, 7]]]],)
print(solve(*example))


<!-- dd:dd-q890 -->

### Problem 890 · independent

Given a batch of channels-LAST images `imgs` as a nested list of shape `(b, h, w, c)`, stack the images vertically in batch order (image 0 on top) and return one image as a nested list of shape `(b·h, w, c)`.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[[0], [1]], [[2], [3]], [[4], [5]], [[6], [7]]]
```


In [ ]:
import torch as t
import einops


def solve(imgs):
    """Batch stacked vertically, channels last."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[[0], [1]], [[2], [3]]], [[[4], [5]], [[6], [7]]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(890)


In [ ]:
#@title 💡 Solution — Problem 890
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(imgs):
    """Batch stacked vertically, channels last."""
    return einops.rearrange(t.tensor(imgs), 'b h w c -> (b h) w c').tolist()


example = ([[[[0], [1]], [[2], [3]]], [[[4], [5]], [[6], [7]]]],)
print(solve(*example))


<!-- dd:dd-q891 -->

### Problem 891 · independent

Given one channels-first image `img` as a nested list of shape `(c, h, w)`, stack its channels vertically — all of channel 0's rows, then all of channel 1's — and return the `(c·h, w)` nested list.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
[[0, 1], [2, 3], [4, 5], [6, 7]]
```


In [ ]:
import torch as t
import einops


def solve(img):
    """Channels stacked vertically."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[0, 1], [2, 3]], [[4, 5], [6, 7]]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(891)


In [ ]:
#@title 💡 Solution — Problem 891
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(img):
    """Channels stacked vertically."""
    return einops.rearrange(t.tensor(img), 'c h w -> (c h) w').tolist()


example = ([[[0, 1], [2, 3]], [[4, 5], [6, 7]]],)
print(solve(*example))


#### Common mistakes

- **"`(h w)` and `(w h)` give the same flattening."** — Same length,
  different order: left-slow/right-fast. The task's phrase "row-major" /
  "column by column" / "X's block first" picks the order for you.
- **"Axes must be adjacent to merge."** — The pattern happily merges
  distant axes ('b h w c -> h (b w) c'); einops inserts the implied
  transpose. By hand you'd have to permute them together first — that
  two-step is exactly what the pattern hides.
- **"Merging loses information."** — It's a pure relabeling; every element
  keeps a unique address. The inverse operation (splitting, next KP)
  recovers the original — provided you remember one of the factor sizes.


<!-- dd:dd-kp-einops-split-axes -->

## Splitting axes with named factors

`einops.split-axes`


Parentheses on the **input** side SPLIT an axis into factors — the exact
inverse of merging:

> `'c (h w) -> c h w', h=h`
> — the length-h·w axis is declared to be h blocks of w, and unpacked.

The new ingredients:

1. **You must tell einops the factor sizes it can't infer.** An axis of
   length 12 could be (h=2, w=6), (3, 4), (4, 3)… — so sizes arrive as
   keyword arguments. Give any factors such that the rest are forced —
   for a two-way split, ONE keyword suffices (`h=3` fixes w = 12/3).
2. **Order inside the parens declares how the axis was PACKED** — left
   slow, right fast, same convention as merging. `(h w)` says "this axis
   is h blocks, each of length w". Splitting with the wrong order doesn't
   error (sizes may still divide) — it unpacks garbage. The task's
   description of how the data was laid out ("row-major tiles", "group
   index slowest/fastest") is the ground truth for the order.
3. **Split and merge combine in one pattern** — the signature einops move.
   `'(b w) c -> b w c'` unpacks; `'h p c -> (h p) c'` repacks;
   `'(h w) p1 p2 c -> (h p1) (w p2) c'` does both at once (that one
   reassembles an image from its tile stack — split the tile index into
   grid coordinates, then merge each with its within-tile axis).

Reading a split-merge pattern: first find every parenthesized group on the
left (what gets unpacked, and in what packing order), then on the right
(what gets packed). The names in the middle just carry through.


Task: restore a flattened image given its height; split a sequence into
chunks; reassemble tiles into an image.


In [ ]:
import torch as t
import einops

# Round-trip: flatten (merge), then restore (split) with one known factor.
img = t.arange(12).reshape(3, 2, 2)                  # (c, h, w)
flat = einops.rearrange(img, 'c h w -> c (h w)')      # (3, 4)
back = einops.rearrange(flat, 'c (h w) -> c h w', h=2)
assert t.equal(back, img)                       # perfect inverse

print("merge then split round-trips:", bool(t.equal(back, img)))




A known segment length is enough to split a packed time axis into (segment, position-within-segment).


In [ ]:
import torch as t
import einops

# Split a sequence into p-token segments: t = n segments of length p.
seq = t.arange(24).reshape(2, 6, 2)                  # (b, t=6, d)
chunks = einops.rearrange(seq, 'b (n p) d -> b n p d', p=3)
assert chunks.shape == (2, 2, 3, 2)
assert t.equal(chunks[0, 0], seq[0, :3])       # first 3 tokens

print("sequence", tuple(seq.shape), "-> chunks", tuple(chunks.shape), "| chunk 0 =", chunks[0, 0].tolist())




Now both at once: unpack the tile index into grid coordinates on the left, and merge each coordinate with its within-tile axis on the right.


In [ ]:
import torch as t
import einops

# Split AND merge at once: tile stack -> image.
# 6 tiles of shape (2, 2), listed row-major from a (2x3)-tile image.
tiles = t.arange(24).reshape(6, 2, 2)                # ((h w), p1, p2)
image = einops.rearrange(tiles, '(h w) p1 p2 -> (h p1) (w p2)', h=2)
assert image.shape == (4, 6)                          # (2*2, 3*2)
# Tile 0 occupies the top-left 2x2 block:
assert image[:2, :2].tolist() == tiles[0].tolist()
print("6 tiles -> one", tuple(image.shape), "image:")
print(image)




Why each step:

1. The round-trip (`back == img`) is the defining property of split-as-
   inverse-of-merge, and doubles as your self-test recipe: whenever a split
   pattern feels shaky, merge it back and compare.
2. In the chunking, `p=3` (not n) is given — either works, and choosing the
   one the task names ("segments of length p") keeps the code aligned with
   the prose.
3. The tile reassembly deserves slow reading: `(h w)` unpacks the tile
   index into grid row/column (row-major, hence h slow); then `(h p1)`
   merges grid-row with within-tile-row. Two coordinate systems zipped
   together — one pattern, no loops, no arithmetic on indices.


<!-- dd:dd-q390 -->

### Problem 390 · faded — your turn

Restore (c, h·w) to (c, h, w), given h.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.]],

        [[4., 5.],
         [6., 7.]]])
```


In [ ]:
import torch as t
import einops

def solve(flat, h):
    """Undo the per-channel flatten: declare how the axis was packed."""
    return einops.rearrange(flat, '_____', h=h)


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(390)


In [ ]:
#@title 💡 Solution — Problem 390
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(flat, h):
    return einops.rearrange(flat, 'c (h w) -> c h w', h=h)


print(solve(t.arange(8.0).reshape(2, 4), 2))


<!-- dd:dd-q315 -->

### Problem 315 · guided

Write a function solve(seq, p) that takes a batch of sequences of shape (b, t, d) — t divisible by p — and splits each sequence into segments of length p: return shape (b, t//p, p, d), the einops pattern 'b (n p) d -> b n p d'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [2., 3.]],

         [[4., 5.],
          [6., 7.]]]])
```


<details>
<summary>Hints</summary>

1. (b, t, d) with t divisible by p → (b, t/p, p, d): the time axis splits
   into (segments × length-p).
2. Which factor does the task hand you? Pass it as the keyword.
3. `'b (n p) d -> b n p d', p=p` — n is inferred.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq, p):
    """Return shape (b, t//p, p, d), the einops pattern 'b (n p) d -> b n p d'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(315)


In [ ]:
#@title 💡 Solution — Problem 315
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq, p):
    return einops.rearrange(seq, 'b (n p) d -> b n p d', p=p)


print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


<!-- dd:dd-q337 -->

### Problem 337 · independent

Write a function solve(merged, b) that takes an image strip of shape (h, b*w, c) — b images concatenated side by side — and SPLITS it back into the batch: return shape (b, h, w, c) via the inverse pattern 'h (b w) c -> b h w c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [2., 3.]]],


        [[[4., 5.],
          [6., 7.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(merged, b):
    """Return shape (b, h, w, c) via the inverse pattern 'h (b w) c -> b h w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(337)


In [ ]:
#@title 💡 Solution — Problem 337
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(merged, b):
    return einops.rearrange(merged, 'h (b w) c -> b h w c', b=b)


print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


<!-- dd:dd-q320 -->

### Problem 320 · independent

Write solve(imgs) for a (B, H, W, C) batch: divide each image into two equal halves along HEIGHT and stack the halves into the batch axis — output (2·B, H/2, W, C) with all the TOP halves first, then all the bottom halves. Pattern: 'b (two h) w c -> (two b) h w c' with two=2.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(320)


In [ ]:
#@title 💡 Solution — Problem 320
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    return einops.rearrange(imgs, 'b (two h) w c -> (two b) h w c', two=2)

print(solve(imgs))


<!-- dd:dd-q331 -->

### Problem 331 · independent

Write a function solve(arr, r) that takes a channels-first batch (b, c, h, w), keeps only the EVEN-indexed images arr[0], arr[2], ... (assume that count is divisible by r), and tiles them into an r-row grid of shape (c, r*h, (count//r)*w) — a slice composed with the grid rearrange '(r n) c h w -> c (r h) (n w)'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 0.,  1.,  2.,  3.],
         [ 8.,  9., 10., 11.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, r):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(4, 1, 1, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(331)


In [ ]:
#@title 💡 Solution — Problem 331
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, r):
    return einops.rearrange(arr[::2], '(r n) c h w -> c (r h) (n w)', r=r)


print(solve(t.arange(16.0).reshape(4, 1, 1, 4), 2))


<!-- dd:dd-q921 -->

### Problem 921 · independent

Given integers `n` and `h` with `h` dividing `n`, return the numbers 0 to n−1 arranged row-major in a grid with `h` rows and n/h columns, as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0, 1], [2, 3], [4, 5]]
```


In [ ]:
import torch as t
import einops


def solve(n, h):
    """0..n-1 into h rows."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (6, 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(921)


In [ ]:
#@title 💡 Solution — Problem 921
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(n, h):
    """0..n-1 into h rows."""
    return einops.rearrange(t.arange(n), '(h w) -> h w', h=h).tolist()


example = (6, 3)
print(solve(*example))


<!-- dd:dd-q924 -->

### Problem 924 · independent

Given integers `start`, `stop` and `w` with `w` dividing stop−start, return the numbers start to stop−1 arranged row-major in a grid with `w` columns, as a nested list of shape ((stop−start)/w, w).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[3, 4], [5, 6], [7, 8]]
```


In [ ]:
import torch as t
import einops


def solve(start, stop, w):
    """start..stop-1 into w columns."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 9, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(924)


In [ ]:
#@title 💡 Solution — Problem 924
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(start, stop, w):
    """start..stop-1 into w columns."""
    return einops.rearrange(t.arange(start, stop), '(h w) -> h w', w=w).tolist()


example = (3, 9, 2)
print(solve(*example))


<!-- dd:dd-q925 -->

### Problem 925 · independent

Given a flat list `vals` and an integer `h` that divides its length, split the list into `h` rows in order (row-major) and return the nested list of shape (h, length/h).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1, 2, 3], [4, 5, 6]]
```


In [ ]:
import torch as t
import einops


def solve(vals, h):
    """Flat list into h rows."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3, 4, 5, 6], 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(925)


In [ ]:
#@title 💡 Solution — Problem 925
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(vals, h):
    """Flat list into h rows."""
    return einops.rearrange(t.tensor(vals), '(h w) -> h w', h=h).tolist()


example = ([1, 2, 3, 4, 5, 6], 2)
print(solve(*example))


<!-- dd:dd-q928 -->

### Problem 928 · independent

Given a grid `grid` as a nested list of shape `(h, w)`, return its values as one flat row-major list of length `h·w`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1, 2, 3, 4]
```


In [ ]:
import torch as t
import einops


def solve(grid):
    """Grid back to a flat list."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(928)


In [ ]:
#@title 💡 Solution — Problem 928
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops


def solve(grid):
    """Grid back to a flat list."""
    return einops.rearrange(t.tensor(grid), 'h w -> (h w)').tolist()


example = ([[1, 2], [3, 4]],)
print(solve(*example))


#### Common mistakes

- **"einops can infer both factors of a split."** — It can infer ONE
  (total ÷ known); the rest are yours to supply as keywords. No keyword,
  no split — the error message will list what's missing.
- **"Wrong paren order in a split will error."** — Only if the sizes fail
  to divide. `(h w)` vs `(w h)` with square-ish factors both "work" and one
  is silently scrambled. The packing order comes from how the data was
  BUILT — reread the task's layout description, then round-trip-test.
- **"Tile reassembly needs index arithmetic."** — The split-merge pattern
  `('(h w) p1 p2 -> (h p1) (w p2)')` IS the index arithmetic, stated
  declaratively. If you're computing offsets by hand around einops, the
  pattern can probably absorb the work.


<!-- dd:dd-kp-einops-singleton-and-lists -->

## Singleton axes and lists as an axis

`einops.singleton-and-lists`


Two small pattern features that finish the rearrange grammar:

**`1` — a literal singleton axis.** Writing `1` in a pattern inserts (on
the right) or consumes (on the left) a length-1 axis:

- `'h w -> 1 h w'` — add a leading channel/batch axis (the einops spelling
  of `x[None]`).
- `'b h w c -> b 1 h w c'` — insert one mid-tensor.
- `'1 h w -> h w'` — squeeze a known singleton, with verification: if that
  axis isn't length 1, einops errors instead of silently squeezing the
  wrong thing.

Keeping a REDUCED axis as a singleton (`'h w c -> 1 w c'` in reduce) also
uses this — the reduce KP picks that up.

**A Python list as the first axis.** Handing einops a LIST of same-shape
tensors makes the list index axis 0 — pattern it like any other axis:

- `einops.rearrange([img_a, img_b], 'b h w c -> h (b w) c')` — two images
  side by side, no explicit t.stack first.
- `'b h w c -> b h w c'` on a list is exactly t.stack: the identity
  pattern, with the list→tensor conversion as the entire point.

Together these subsume t.stack / t.unsqueeze / t.squeeze with the
same pattern language you're already using — one notation for the whole
shape-plumbing toolbox.


Task: stack a list of images into a batch; insert a singleton axis into a
plain tensor; stack a list AND merge in one pattern; squeeze with a check.


In [ ]:
import torch as t
import einops

imgs = [t.ones((2, 3, 1)) * i for i in range(4)]   # list of (h, w, c)

# List -> batch axis: the identity pattern DOES the stacking.
batch = einops.rearrange(imgs, 'b h w c -> b h w c')
assert batch.shape == (4, 2, 3, 1)
assert batch[2, 0, 0, 0] == 2.0                      # list order preserved

print("list of 4 images -> batch", tuple(batch.shape), "| image 2's first pixel:", batch[2, 0, 0, 0].item())




Stacking made a batch out of several tensors; a singleton adds a length-1 axis to ONE tensor without copying anything.


In [ ]:
import torch as t
import einops

# Singleton insertion: a plain 2-D tensor gains a leading axis.
x2d = t.arange(6).reshape(2, 3)
x3d = einops.rearrange(x2d, 'h w -> 1 h w')
assert x3d.shape == (1, 2, 3)

print("singleton insert:", tuple(x2d.shape), "->", tuple(x3d.shape))




The list-to-batch conversion composes with any merge in the same pattern.


In [ ]:
import torch as t
import einops

# Both at once: stack a list AND lay the images out side by side.
pair = [t.zeros((2, 2, 1)), t.ones((2, 2, 1))]
wide = einops.rearrange(pair, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 1)
assert wide[0, :, 0].tolist() == [0.0, 0.0, 1.0, 1.0]  # a then b, left to right

print("stack + side by side:", tuple(wide.shape), "| row 0 =", wide[0, :, 0].tolist())




Removing a singleton is the reverse pattern, and einops checks that the `1` is really there.


In [ ]:
import torch as t
import einops

# Squeeze with verification: consuming a '1' that isn't there fails loudly.
try:
    einops.rearrange(x2d, '1 h w -> h w')            # x2d is 2-D — no 1 axis
    raised = False
except Exception as err:
    raised = True
    print("squeezing an axis that isn't there ->", type(err).__name__)
assert raised

print("all four moves ran; squeeze raised:", raised)




Why each step:

1. The identity pattern on a list looks like a no-op and isn't — the
   conversion is the operation. Reading einops code, remember the input
   TYPE is part of the semantics.
2. The combined example is the idiom to keep: list-stack + merge in one
   declarative step replaces stack-then-rearrange chains. Note b defaults
   to slow in `(b w)`: first list element leftmost.
3. The verified squeeze failing on 2-D data is einops' shape checking
   again — `'1 h w -> h w'` documents an EXPECTATION about the input, and
   the library enforces it. `t.squeeze` would have silently done something.


<!-- dd:dd-q361 -->

### Problem 361 · faded — your turn

A Python list of (h, w, c) images → one (b, h, w, c) batch.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[1.],
          [1.]],

         [[1.],
          [1.]]],


        [[[0.],
          [0.]],

         [[0.],
          [0.]]]])
```


In [ ]:
import torch as t
import einops

def solve(imgs):
    """Stack the list: the list index becomes axis b."""
    return einops.rearrange(imgs, '_____')


# Example run — the grader calls solve() with several inputs.
print(solve([t.ones((2, 2, 1)), t.zeros((2, 2, 1))]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(361)


In [ ]:
#@title 💡 Solution — Problem 361
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(imgs):
    return einops.rearrange(imgs, 'b h w c -> b h w c')


print(solve([t.ones((2, 2, 1)), t.zeros((2, 2, 1))]))


<!-- dd:dd-q360 -->

### Problem 360 · guided

Write a function solve(x2d) that takes a plain 2-D tensor (h, w) and returns the 3-D version (1, h, w) with a leading singleton channel axis — 'h w -> () h w'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 1., 1.],
         [1., 1., 1.]]])
```


<details>
<summary>Hints</summary>

1. (h, w) → (1, h, w): nothing moves; one axis appears.
2. The literal `1` on the output side inserts it.
3. `'h w -> 1 h w'`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x2d):
    """Return the 3-D version (1, h, w) with a leading singleton channel axis — 'h w -> () h w'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 3))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(360)


In [ ]:
#@title 💡 Solution — Problem 360
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x2d):
    return einops.rearrange(x2d, 'h w -> () h w')


print(solve(t.ones((2, 3))))


<!-- dd:dd-q358 -->

### Problem 358 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and INSERTS a singleton axis right after the batch axis: return shape (b, 1, h, w, c) via the '()' output group — 'b h w c -> b () h w c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[[1.],
           [1.]],

          [[1.],
           [1.]]]],



        [[[[1.],
           [1.]],

          [[1.],
           [1.]]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b, 1, h, w, c) via the '()' output group — 'b h w c -> b () h w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 2, 1))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(358)


In [ ]:
#@title 💡 Solution — Problem 358
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b h w c -> b () h w c')


print(solve(t.ones((2, 2, 2, 1))))


<!-- dd:dd-q374 -->

### Problem 374 · independent

Write a function solve(imgs) that takes a PYTHON LIST of channels-last images (h, w, c) and concatenates them SIDE BY SIDE: return shape (h, len(list)*w, c) — the list index merging straight into width via 'b h w c -> h (b w) c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.],
         [1.]],

        [[0.],
         [1.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(imgs):
    """Return shape (h, len(list)*w, c) — the list index merging straight into width via 'b h w c -> h (b w) c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.zeros((2, 1, 1)), t.ones((2, 1, 1))]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(374)


In [ ]:
#@title 💡 Solution — Problem 374
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(imgs):
    return einops.rearrange(imgs, 'b h w c -> h (b w) c')


print(solve([t.zeros((2, 1, 1)), t.ones((2, 1, 1))]))


<!-- dd:dd-q376 -->

### Problem 376 · independent

Write solve(img_a, img_b) for two identical-shape (H, W, C) images: place them SIDE BY SIDE into one (H, 2·W, C) image, img_a on the left. Pattern: einops.rearrange([img_a, img_b], 'b h w c -> h (b w) c').

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]

def solve(img_a, img_b):
    # Write your solution here
    return None

print(solve(img_a, img_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(376)


In [ ]:
#@title 💡 Solution — Problem 376
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]

def solve(img_a, img_b):
    return einops.rearrange([img_a, img_b], 'b h w c -> h (b w) c')

print(solve(img_a, img_b))


<!-- dd:dd-q333 -->

### Problem 333 · independent

Write a function solve(tensors) that takes a PYTHON LIST of channels-first images, each of shape (c, h, w), and returns a single channels-LAST batch tensor of shape (len(list), h, w, c) — einops.rearrange applied directly to the list stacks it as the new leading axis.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[1.],
          [1.]],

         [[1.],
          [1.]]],


        [[[0.],
          [0.]],

         [[0.],
          [0.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    """Return a single channels-LAST batch array of shape (len(list), h, w, c) — einops.rearrange applied directly to"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(333)


In [ ]:
#@title 💡 Solution — Problem 333
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    return einops.rearrange(tensors, 'b c h w -> b h w c')


print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


<!-- dd:dd-q334 -->

### Problem 334 · independent

Write solve(img_a, img_b) for two identical-shape (H, W, C) images: produce a single (2·H, W, C) image whose rows INTERLEAVE the two inputs — row 0 of a, row 0 of b, row 1 of a, row 1 of b, … Pattern: einops.rearrange([img_a, img_b], 'b h w c -> (h b) w c').

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 155, 205],
         [255, 155, 205],
         [255, 155, 205],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 155, 205],
         [255, 155, 205],
         [255, 155, 205],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 155, 205],
         [255, 155, 205],
         [255, 155, 205],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[1]
img_b = arr[1]

def solve(img_a, img_b):
    # Write your solution here
    return None

print(solve(img_a, img_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(334)


In [ ]:
#@title 💡 Solution — Problem 334
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[1]
img_b = arr[1]

def solve(img_a, img_b):
    return einops.rearrange([img_a, img_b], 'b h w c -> (h b) w c')

print(solve(img_a, img_b))


<!-- dd:dd-q365 -->

### Problem 365 · independent

Write a function solve(tensors) that takes a PYTHON LIST of channels-first images (c, h, w) and returns a single tensor of shape (h, w, c, len(list)) — stacked AND rearranged so the list axis lands LAST: rearrange(list, 'b c h w -> h w c b').

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[1., 0.]],

         [[1., 0.]]],


        [[[1., 0.]],

         [[1., 0.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    """Return a single array of shape (h, w, c, len(list)) — stacked AND rearranged so the list axis lands LAST."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(365)


In [ ]:
#@title 💡 Solution — Problem 365
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    return einops.rearrange(tensors, 'b c h w -> h w c b')


print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


#### Common mistakes

- **"I must t.stack a list before einops can touch it."** — A list of
  same-shape tensors is accepted directly; its index becomes the first
  axis. The stack is the pattern's job.
- **"`1` in a pattern is a size I'm asserting for a normal axis."** — It's
  a LITERAL singleton: inserted on the right, consumed (with verification)
  on the left. Naming it (like 'c') instead would bind a real axis.
- **"Squeezing with einops is overkill — t.squeeze is fine."** —
  t.squeeze removes ALL singletons, including ones you didn't expect
  (a batch that happens to be size 1!). `'1 h w -> h w'` removes exactly
  the declared one and errors otherwise — overkill is the feature.


<!-- dd:dd-kp-einops-grids-montage -->

## Laying batches out as grids

`einops.grids-montage`


"Tile these b images into a g1-row grid" — the montage — is the flagship
split-then-merge pattern:

> `'(g1 g2) h w c -> (g1 h) (g2 w) c', g1=rows`

Read it in two moves:

1. **Split the batch into grid coordinates.** `(g1 g2)` on the left declares
   the batch axis packs g1 rows of g2 images, ROW-MAJOR (g1 slow: images
   0..g2-1 form grid row 0). One keyword fixes both factors.
2. **Merge each grid coordinate with its image dimension.** `(g1 h)`: grid
   row with within-image height — grid row 0's images occupy output rows
   0..h-1. `(g2 w)`: grid column with width. The output is one big
   ((g1·h) × (g2·w)) image.

Every montage variant is a small edit to this template:

- Channels-first data: same idea around the c axis —
  `'(g1 g2) c h w -> c (g1 h) (g2 w)'`.
- A single row of images ("side by side") is the degenerate g1=1 case —
  which collapses to the merge-KP pattern `'b h w c -> h (b w) c'`.
- Column-major filling would be `(g2 g1)` on the left — the packing-order
  question from the split KP, again decided by the task's words
  ("row-major", "grid position (i, j) holds image i·cols + j").

The montage also runs in REVERSE — carving a grid image back into a batch —
by swapping the pattern's sides: `'(g1 h) (g2 w) c -> (g1 g2) h w c'` with
two keywords, since neither factor of each merged axis is inferable alone.


Task: six images into a 3-row × 2-column grid, row-major — verified by
locating specific images.


In [ ]:
import torch as t
import einops

# Six 2x2 single-channel images; image k is constant k (easy to locate).
imgs = t.stack([t.full((2, 2, 1), float(k)) for k in range(6)])
assert imgs.shape == (6, 2, 2, 1)

grid = einops.rearrange(imgs, '(g1 g2) h w c -> (g1 h) (g2 w) c', g1=3)
assert grid.shape == (6, 4, 1)               # (3*2, 2*2, 1)

# Row-major placement: grid row 0 holds images 0,1; row 1 -> 2,3; row 2 -> 4,5.
assert grid[0, 0, 0] == 0.0                  # top-left block = image 0
assert grid[0, 2, 0] == 1.0                  # top-right block = image 1
assert grid[2, 0, 0] == 2.0                  # second row starts image 2
assert grid[4, 2, 0] == 5.0                  # bottom-right = image 5

print("6 images", tuple(imgs.shape), "-> montage", tuple(grid.shape))
print(grid[:, :, 0])          # each image is a constant block, so read them off




The montage runs in reverse by swapping the pattern's sides — but now each merged axis hides two unknowns, so each group needs its own keyword.


In [ ]:
import torch as t
import einops

# Reverse: carve the montage back into the batch. Each merged input axis
# hides two unknowns, so each group needs one keyword: g1 (fixes h) AND
# g2 (fixes w).
back = einops.rearrange(grid, '(g1 h) (g2 w) c -> (g1 g2) h w c', g1=3, g2=2)
assert t.equal(back, imgs)
print("carved back to the batch exactly:", bool(t.equal(back, imgs)))




Why each step:

1. Constant-valued test images turn placement checking into value lookups:
   `grid[0, 2]` sitting in grid-row 0, grid-col 1 must equal image 1 under
   row-major packing. Build such fixtures whenever a layout task confuses
   you — arange or constants, never random.
2. The g1=3 keyword does double duty: fixes g2=2 AND documents "3 rows" —
   matching the task's phrasing decides WHICH factor you pass.
3. The reverse pattern needs a keyword PER GROUP (g1 and g2) because each
   merged input axis hides two unknowns and einops solves exactly one
   unknown per parenthesized group.


<!-- dd:dd-q389 -->

### Problem 389 · faded — your turn

Six images → 3×2 grid, row-major.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],
… (truncated)
```


In [ ]:
import torch as t
import einops

def solve(imgs):
    """(6, h, w, c) -> ((3h), (2w), c), images placed row-major."""
    return einops.rearrange(imgs, '_____', g1=3)


_base = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = t.cat([_base] * (-(-6 // _base.shape[0])))[:6]


print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(389)


In [ ]:
#@title 💡 Solution — Problem 389
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

_base = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = t.cat([_base] * (-(-6 // _base.shape[0])))[:6]

def solve(imgs):
    return einops.rearrange(imgs, '(g1 g2) h w c -> (g1 h) (g2 w) c', g1=3)

print(solve(imgs))


<!-- dd:dd-q364 -->

### Problem 364 · guided

Write a function solve(arr, b1) that takes a channels-LAST batch (b, h, w, c) — b divisible by b1 — and tiles the images into a b1-row montage: return shape (b1*h, (b//b1)*w, c) via '(b1 b2) h w c -> (b1 h) (b2 w) c'. (The companion drill does this for channels-first input.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.],
         [1.],
         [2.],
         [3.]],

        [[4.],
         [5.],
         [6.],
         [7.]]])
```


<details>
<summary>Hints</summary>

1. A channels-LAST batch (b, h, w, c), b divisible by b1, tiled into a
   b1-row montage — the template with different names.
2. Which factor is given (b1 rows), which inferred?
3. `'(b1 b2) h w c -> (b1 h) (b2 w) c', b1=b1`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, b1):
    """Return shape (b1*h, (b//b1)*w, c) via '(b1 b2) h w c -> (b1 h) (b2 w) c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(4, 1, 2, 1), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(364)


In [ ]:
#@title 💡 Solution — Problem 364
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, b1):
    return einops.rearrange(arr, '(b1 b2) h w c -> (b1 h) (b2 w) c', b1=b1)


print(solve(t.arange(8.0).reshape(4, 1, 2, 1), 2))


<!-- dd:dd-q329 -->

### Problem 329 · independent

Write a function solve(arr, g1) that takes a channels-first batch of shape (b, c, h, w) — b divisible by g1 — and tiles the images into a g1-row grid: return shape (c, g1*h, (b//g1)*w) via '(g1 g2) c h w -> c (g1 h) (g2 w)', batch order filling each row left to right.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1., 2., 3.],
         [4., 5., 6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, g1):
    """Return shape (c, g1*h, (b//g1)*w) via '(g1 g2) c h w -> c (g1 h) (g2 w)', batch order filling each row left to"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(4, 1, 1, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(329)


In [ ]:
#@title 💡 Solution — Problem 329
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, g1):
    return einops.rearrange(arr, '(g1 g2) c h w -> c (g1 h) (g2 w)', g1=g1)


print(solve(t.arange(8.0).reshape(4, 1, 1, 2), 2))


<!-- dd:dd-q382 -->

### Problem 382 · independent

Write solve(imgs) for a (6, H, W, C) channels-last batch: split the batch into a 3-row × 2-column grid, row-major, producing one ((3·H), (2·W), C) image. Pattern: '(r nc) h w ch -> (r h) (nc w) ch' with r=3.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 155, 205],
         [255, 155, 205],
         [255, 155, 205]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(382)


In [ ]:
#@title 💡 Solution — Problem 382
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    return einops.rearrange(imgs, '(r nc) h w ch -> (r h) (nc w) ch', r=3)

print(solve(imgs))


<!-- dd:dd-q318 -->

### Problem 318 · independent

Write solve(imgs) for a (12, C, H, W) batch of channels-first images: tile them into a single CHW image laid out as a 4-row × 3-column grid (row-major: image i lands at row i//3, column i%3). Pattern: '(g1 g2) c h w -> c (g1 h) (g2 w)' with g1=4.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 255, 255,  ..., 205, 205, 205],
         [255, 255, 255,  ..., 205, 205, 205],
         [255, 255, 255,  ..., 205, 205, 205],
         ...,
         [205, 205, 205,  ..., 155, 155, 155],
         [205, 205, 205,  ..., 155, 155, 155],
         [205, 205, 205,  ..., 155, 155, 155]],

        [[205, 205, 205,  ..., 255, 255, 255],
         [205, 205, 205,  ..., 255, 255, 255],
         [205, 205, 205,  ..., 255, 255, 255],
         ...,
         [155, 155, 155,  ..., 205, 205, 205],
         [155, 155, 155,  ..., 205, 205, 205],
         [155, 155, 155,  ..., 205, 205, 205]],

        [[155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         ...,
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255]]], dtype=torch.uint8)
```


In [ ]:
import torch as t
import einops

_base = t.tensor(np.load('/delta_numbers.npy'))
imgs = t.cat([_base] * (-(-12 // _base.shape[0])))[:12]

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(318)


In [ ]:
#@title 💡 Solution — Problem 318
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

_base = t.tensor(np.load('/delta_numbers.npy'))
imgs = t.cat([_base] * (-(-12 // _base.shape[0])))[:12]

def solve(imgs):
    return einops.rearrange(imgs, '(g1 g2) c h w -> c (g1 h) (g2 w)', g1=4)

print(solve(imgs))


<!-- dd:dd-q381 -->

### Problem 381 · independent

Write solve(img_a, img_b, img_c, img_d) for four identical-shape (H, W, C) images: combine them into one (2·H, 2·W, C) image laid out as a 2×2 grid, row-major (a top-left, b top-right, c bottom-left, d bottom-right). Pattern: einops.rearrange([a, b, c, d], '(r nc) h w ch -> (r h) (nc w) ch', r=2).

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]
img_c = arr[0]
img_d = arr[0]

def solve(img_a, img_b, img_c, img_d):
    # Write your solution here
    return None

print(solve(img_a, img_b, img_c, img_d))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(381)


In [ ]:
#@title 💡 Solution — Problem 381
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]
img_c = arr[0]
img_d = arr[0]

def solve(img_a, img_b, img_c, img_d):
    return einops.rearrange([img_a, img_b, img_c, img_d], '(r nc) h w ch -> (r h) (nc w) ch', r=2)

print(solve(img_a, img_b, img_c, img_d))


<!-- dd:dd-q322 -->

### Problem 322 · independent

Write solve(x, hs, ws) for a (B, C, H, W) batch and integer subgrid factors: carve each image's height into hs strips and width into ws strips, moving the (hs, ws) subgrid indices OUT into the batch axis — output (hs·ws·B, C, H/hs, W/ws), subgrid-major then batch. Pattern: 'b c (h hs) (w ws) -> (hs ws b) c h w'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          ...,
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255],
          [255, 255, 255,  ..., 255, 255, 255]],

         [[205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          ...,
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205],
          [205, 205, 205,  ..., 205, 205, 205]],

         [[155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          ...,
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155],
          [155, 155, 155,  ..., 155, 155, 155]]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
x = arr[:2]
hs = 2
ws = 2

def solve(x, hs, ws):
    # Write your solution here
    return None

print(solve(x, hs, ws))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(322)


In [ ]:
#@title 💡 Solution — Problem 322
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
x = arr[:2]
hs = 2
ws = 2

def solve(x, hs, ws):
    return einops.rearrange(x, 'b c (h hs) (w ws) -> (hs ws b) c h w', hs=hs, ws=ws)

print(solve(x, hs, ws))


<!-- dd:dd-q371 -->

### Problem 371 · independent

Write a function solve(y, hs, ws) that takes a tensor of shape (hs*ws*b, c, h, w) — spatial subgrids packed into the batch axis, subgrid index slowest — and UNPACKS them back into space: return shape (b, c, h*hs, w*ws) via '(hs ws b) c h w -> b c (h hs) (w ws)'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [2., 3.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(y, hs, ws):
    """Return shape (b, c, h*hs, w*ws) via '(hs ws b) c h w -> b c (h hs) (w ws)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(4, 1, 1, 1), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(371)


In [ ]:
#@title 💡 Solution — Problem 371
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(y, hs, ws):
    return einops.rearrange(y, '(hs ws b) c h w -> b c (h hs) (w ws)', hs=hs, ws=ws)


print(solve(t.arange(4.0).reshape(4, 1, 1, 1), 2, 2))


<!-- dd:dd-q531 -->

### Problem 531 · independent

Write a function solve(imgs, rows) that takes a channels-last batch of images with shape (b, h, w, c) and a grid row count, and returns the single montage image of shape ((rows*h), (cols*w), c) holding all b images — filled COLUMN-major, so image k lands at grid row k % rows and grid column k // rows. b is divisible by rows. This is the row-major montage pattern with one edit, and the edit is on the INPUT side: which of the two batch factors counts slowly decides the filling order, so swap the split order rather than renaming anything.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[0., 0., 3., 3.],
        [0., 0., 3., 3.],
        [1., 1., 4., 4.],
        [1., 1., 4., 4.],
        [2., 2., 5., 5.],
        [2., 2., 5., 5.]])
```


In [ ]:
import torch as t
import einops

def solve(imgs, rows):
    """(b, h, w, c) -> ((rows*h), (cols*w), c), images filled COLUMN-major."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.stack([t.full((2, 2, 1), float(k)) for k in range(6)]), 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(531)


In [ ]:
#@title 💡 Solution — Problem 531
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(imgs, rows):
    """(b, h, w, c) -> ((rows*h), (cols*w), c), images filled COLUMN-major."""
    return einops.rearrange(imgs, '(g2 g1) h w c -> (g1 h) (g2 w) c', g1=rows)


example = (t.stack([t.full((2, 2, 1), float(k)) for k in range(6)]), 3)
print(solve(*example)[:, :, 0])


#### Common mistakes

- **"Montages need a loop pasting images into a canvas."** — The
  split-merge pattern is the whole operation. If you're computing paste
  offsets, the pattern replaces that code.
- **"g1 is rows because it's named g1."** — It's rows because it MERGES
  WITH `h` on the right. Its position in the split on the left decides the
  fill order (slow = row-major, fast = column-major), not what it means.
  Rename freely; position does the work. (Corollary: to fill column-major, swap the split order, not the
  names.)
- **"The grid pattern only works for images."** — Any batch × per-item-2D
  data montages the same way; and the same split-merge shape reappears in
  patches, space-to-depth, and pooling. Learn it as geometry, not as an
  image trick.


<!-- dd:dd-kp-einops-patches-space-depth -->

## Patches, space-to-depth, depth-to-space

`einops.patches-space-depth`


Vision architectures constantly trade SPACE for DEPTH (or batch): cut each
image into little patches and treat them as tokens (ViT), fold pixel blocks
into channels (space-to-depth), or unfold them back (depth-to-space /
pixel-shuffle). All are the grid pattern of the last KP with the roles
recast:

**Patch extraction** — split each spatial axis into (blocks × within-block),
then pull the block coordinates out as a patch index:

> `'(h p1) (w p2) c -> (h w) p1 p2 c', p1=P, p2=P`

Height splits into h blocks of p1 rows; width likewise; merging (h w)
row-major gives the patch list. **Reassembly is the same pattern reversed**
— this page's faded exercise (q323) practises exactly that direction,
depth-to-space's cousin.

**Space-to-depth** — the within-block offsets fold into the CHANNEL axis
(the block's grid position stays spatial) instead of becoming a patch index:

> `'b c (h p) (w q) -> b (c p q) h w', p=P, q=P`

Each p×q pixel block's values become extra channels; spatial dims shrink by
P, channels grow by P². (The output channel packing order — c slow, then p,
then q — is dictated by the paren order; tasks specify theirs.)

**Depth-to-space** mirrors it: `'b (c p q) h w -> b c (h p) (w q)'` — the
channel axis DECLARES its factorization, and blocks unfold back into space.
This is pixel-shuffle upsampling.

The discipline for all of them: write the INPUT side to describe how the
data is actually packed (which factor is slow), the OUTPUT side to describe
what the task wants — then hand einops the block sizes. When the packing
order is ambiguous in your head, build a tiny arange example and round-trip.


Task: extract 2×2 patches from an image, reassemble them, and run a
space-to-depth.


In [ ]:
import torch as t
import einops

img = t.arange(16.0).reshape(4, 4, 1)     # (H, W, c=1), values = positions

# PATCHES: split H into 2 blocks of 2, W likewise; block coords -> patch axis.
patches = einops.rearrange(img, '(h p1) (w p2) c -> (h w) p1 p2 c', p1=2, p2=2)
assert patches.shape == (4, 2, 2, 1)
# Patch 0 = top-left 2x2 block, row-major patch order:
assert patches[0, :, :, 0].tolist() == [[0.0, 1.0], [4.0, 5.0]]
assert patches[1, 0, 0, 0] == 2.0          # patch 1 starts at column 2

print("image", tuple(img.shape), "-> patches", tuple(patches.shape))
print("patch 0 =", patches[0, :, :, 0].tolist(), "| patch 1 starts at", patches[1, 0, 0, 0].item())




Reassembly is the extraction pattern with its sides swapped; the keyword moves to whichever side now has the unknowns.


In [ ]:
import torch as t
import einops

# REASSEMBLE: the same pattern, sides swapped (this is q323's shape).
back = einops.rearrange(patches, '(h w) p1 p2 c -> (h p1) (w p2) c', h=2)
assert t.equal(back, img)

print("reassembled to the original:", bool(t.equal(back, img)))




Space-to-depth keeps the block's grid position on the spatial axes and folds the position WITHIN the block into the channel axis.


In [ ]:
import torch as t
import einops

# SPACE-TO-DEPTH on a batch: blocks fold into channels; H, W halve.
x = t.arange(32.0).reshape(1, 2, 4, 4)    # (b, c=2, H, W)
s2d = einops.rearrange(x, 'b c (h p) (w q) -> b (c p q) h w', p=2, q=2)
assert s2d.shape == (1, 8, 2, 2)           # channels x4, spatial /2
# The new channel block for output pixel (0,0) holds input block [0:2, 0:2]:
assert s2d[0, :4, 0, 0].tolist() == [0.0, 1.0, 4.0, 5.0]
print("space-to-depth", tuple(x.shape), "->", tuple(s2d.shape),
      "(channels x4, spatial /2)")
print("pixel (0,0)'s new channels:", s2d[0, :4, 0, 0])




Why each step:

1. Position-valued pixels make each check readable: patch 1 starting at
   value 2.0 confirms row-major patch order and a correct width split.
   This fixture technique is how to debug ANY packing dispute with einops.
2. The reassembly line being the extraction line reversed (with the
   keyword moving to the other side's unknowns) cements the symmetry —
   patches/grids/s2d are one bijection family, direction chosen by which
   side carries the parens you're UNPACKING.
3. In space-to-depth, verify the channel packing: c slow, p, then q — the
   four values 0,1,4,5 are block (0,0) in row-major order, sitting after
   channel 0's... here c=... the first 4 output channels come from input
   channel 0. Reading packed channel layouts element-by-element once
   inoculates against the classic s2d ordering bug.


<!-- dd:dd-q323 -->

### Problem 323 · faded — your turn

Reassemble a row-major tile stack into the image.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 0.],
         [ 1.],
         [ 4.],
         [ 5.]],

        [[ 2.],
         [ 3.],
         [ 6.],
         [ 7.]],

        [[ 8.],
         [ 9.],
         [12.],
         [13.]],

        [[10.],
         [11.],
         [14.],
         [15.]]])
```


In [ ]:
import torch as t
import einops

def solve(patches, h, w):
    """(h*w, p1, p2, c) tiles, row-major -> ((h p1), (w p2), c) image."""
    return einops.rearrange(patches, '_____', h=h, w=w)


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(4, 2, 2, 1), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(323)


In [ ]:
#@title 💡 Solution — Problem 323
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(patches, h, w):
    return einops.rearrange(patches, '(h w) p1 p2 c -> (h p1) (w p2) c', h=h, w=w)


print(solve(t.arange(16.0).reshape(4, 2, 2, 1), 2, 2))


<!-- dd:dd-q313 -->

### Problem 313 · guided

Write a function solve(x, p) that takes a 4-D channels-first batch x of shape (b, c, h, w) — h and w divisible by p — and performs SPACE-TO-DEPTH with block size p: pack each non-overlapping p x p spatial block into the channel dimension. The result has shape (b, c*p*p, h//p, w//p), matching einops.rearrange(x, 'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=p, p2=p).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[ 0.,  2.],
          [ 8., 10.]],

         [[ 1.,  3.],
          [ 9., 11.]],

         [[ 4.,  6.],
          [12., 14.]],

         [[ 5.,  7.],
          [13., 15.]]]])
```


<details>
<summary>Hints</summary>

1. Space-to-depth on (b, c, h, w) with block size p: spatial dims shrink by
   p, channels multiply by p².
2. Split each spatial axis into (blocks × p); fold the two p-factors into
   the channel group — the task states the required channel packing order.
3. `'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=p, p2=p` — or the order the
   prompt demands; verify one block.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, p):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(313)


In [ ]:
#@title 💡 Solution — Problem 313
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, p):
    return einops.rearrange(x, 'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=p, p2=p)


print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


<!-- dd:dd-q401 -->

### Problem 401 · independent

Write solve(img_hwc) for an (H, W, C) image with H and W divisible by 3: extract all non-overlapping 3×3 patches into a ((H/3 · W/3), 3, 3, C) tensor, patches ordered row-major. Pattern: '(h p1) (w p2) c -> (h w) p1 p2 c' with p1=3, p2=3.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]]],


        [[[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_hwc = arr[0]

def solve(img_hwc):
    # Write your solution here
    return None

print(solve(img_hwc))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(401)


In [ ]:
#@title 💡 Solution — Problem 401
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_hwc = arr[0]

def solve(img_hwc):
    return einops.rearrange(img_hwc, '(h p1) (w p2) c -> (h w) p1 p2 c', p1=3, p2=3)

print(solve(img_hwc))


<!-- dd:dd-q404 -->

### Problem 404 · independent

Write a function solve(img, p) that takes a channels-first image (c, h*p, w*p) and EXTRACTS its non-overlapping p x p patches: return shape (h*w, c, p, p) — patches listed row-major — via 'c (h p1) (w p2) -> (h w) c p1 p2'. (The inverse of patch reassembly.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[ 0.,  1.],
          [ 4.,  5.]]],


        [[[ 2.,  3.],
          [ 6.,  7.]]],


        [[[ 8.,  9.],
          [12., 13.]]],


        [[[10., 11.],
          [14., 15.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, p):
    """Return shape (h*w, c, p, p) — patches listed row-major — via 'c (h p1) (w p2) -> (h w) c p1 p2'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 4, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(404)


In [ ]:
#@title 💡 Solution — Problem 404
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, p):
    return einops.rearrange(img, 'c (h p1) (w p2) -> (h w) c p1 p2', p1=p, p2=p)


print(solve(t.arange(16.0).reshape(1, 4, 4), 2))


<!-- dd:dd-q343 -->

### Problem 343 · independent

Write a function solve(arr, p1, p2) that takes a tensor of shape (b, c*p1*p2, h, w) — channel groups ordered with c slowest — and performs DEPTH-TO-SPACE: return shape (b, c, h*p1, w*p2) via 'b (c p1 p2) h w -> b c (h p1) (w p2)', spreading each group of p1*p2 channels over a finer spatial grid.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [2., 3.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, p1, p2):
    """Return shape (b, c, h*p1, w*p2) via 'b (c p1 p2) h w -> b c (h p1) (w p2)', spreading each group of p1*p2 chan"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 4, 1, 1), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(343)


In [ ]:
#@title 💡 Solution — Problem 343
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, p1, p2):
    return einops.rearrange(arr, 'b (c p1 p2) h w -> b c (h p1) (w p2)', p1=p1, p2=p2)


print(solve(t.arange(4.0).reshape(1, 4, 1, 1), 2, 2))


<!-- dd:dd-q350 -->

### Problem 350 · independent

Write solve(img) for an (H, W, C) image with H and W even: within every non-overlapping 2×2 patch, TRANSPOSE the patch (swap its within-patch row and column). Pattern: '(h p1) (w p2) c -> (h p2) (w p1) c' with p1=2, p2=2. Output shape is unchanged; content moves.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(350)


In [ ]:
#@title 💡 Solution — Problem 350
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    return einops.rearrange(img, '(h p1) (w p2) c -> (h p2) (w p1) c', p1=2, p2=2)

print(solve(img))


<!-- dd:dd-q321 -->

### Problem 321 · independent

Write a function solve(x, h1, w1) that takes a tensor of shape (b, h1*w1*c, h, w) and redistributes channel groups into space: return shape (b, c, h*h1, w*w1) via 'b (h1 w1 c) h w -> b c (h h1) (w w1)' — each of the h1*w1 channel groups becomes one sub-position of an enlarged pixel grid.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [2., 3.],
          [4., 5.],
          [6., 7.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, h1, w1):
    """Return shape (b, c, h*h1, w*w1) via 'b (h1 w1 c) h w -> b c (h h1) (w w1)' — each of the h1*w1 channel groups """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 8, 1, 1), 4, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(321)


In [ ]:
#@title 💡 Solution — Problem 321
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, h1, w1):
    return einops.rearrange(x, 'b (h1 w1 c) h w -> b c (h h1) (w w1)', h1=h1, w1=w1)


print(solve(t.arange(8.0).reshape(1, 8, 1, 1), 4, 2))


<!-- dd:dd-q395 -->

### Problem 395 · independent

Write solve(x, h1, w1) for a (B, C, H, W) batch: space-to-depth — carve each spatial map into h1×w1 blocks and fold the block-position indices INTO the channel axis, output (B, h1·w1·C, H/h1, W/w1) with channel order (h1, w1, c). Pattern: 'b c (h h1) (w w1) -> b (h1 w1 c) h w'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[   0,    2,    4,    6],
          [  32,   34,   36,   38]],

         [[  64,   66,   68,   70],
          [  96,   98,  100,  102]],

         [[ 128,  130,  132,  134],
          [ 160,  162,  164,  166]],

         ...,

         [[ 601,  603,  605,  607],
          [ 633,  635,  637,  639]],

         [[ 665,  667,  669,  671],
          [ 697,  699,  701,  703]],

         [[ 729,  731,  733,  735],
          [ 761,  763,  765,  767]]],


        [[[ 768,  770,  772,  774],
          [ 800,  802,  804,  806]],
… (truncated)
```


In [ ]:
import torch as t
import einops

x = t.arange(2 * 12 * 8 * 8).reshape(2, 12, 8, 8)
h1 = 4
w1 = 2

def solve(x, h1, w1):
    # Write your solution here
    return None

print(solve(x, h1, w1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(395)


In [ ]:
#@title 💡 Solution — Problem 395
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

x = t.arange(2 * 12 * 8 * 8).reshape(2, 12, 8, 8)
h1 = 4
w1 = 2

def solve(x, h1, w1):
    return einops.rearrange(x, 'b c (h h1) (w w1) -> b (h1 w1 c) h w', h1=h1, w1=w1)

print(solve(x, h1, w1))


<!-- dd:dd-q398 -->

### Problem 398 · independent

Write a function solve(patches, h, w) that takes a patch stack of shape (h*w, c, p1, p2) — channels-FIRST patches listed row-major — and reassembles the image in channels-first form: return shape (c, h*p1, w*p2) via '(h w) c p1 p2 -> c (h p1) (w p2)'. (The companion drill reassembles channels-last patches.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 0.,  1.,  4.,  5.],
         [ 2.,  3.,  6.,  7.],
         [ 8.,  9., 12., 13.],
         [10., 11., 14., 15.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(patches, h, w):
    """Return shape (c, h*p1, w*p2) via '(h w) c p1 p2 -> c (h p1) (w p2)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(4, 1, 2, 2), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(398)


In [ ]:
#@title 💡 Solution — Problem 398
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(patches, h, w):
    return einops.rearrange(patches, '(h w) c p1 p2 -> c (h p1) (w p2)', h=h, w=w)


print(solve(t.arange(16.0).reshape(4, 1, 2, 2), 2, 2))


<!-- dd:dd-q403 -->

### Problem 403 · independent

Write a function solve(arr, w4) that takes a BHWC batch (b, h, w*w4, c) and folds width chunks into height: factor the width axis as (w, w4) with w4 INNER, then merge w4 into height AHEAD of h — return shape (b, w4*h, w, c) via 'b h (w w4) c -> b (w4 h) w c'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[ 0.,  1.],
          [ 8.,  9.]],

         [[ 2.,  3.],
          [10., 11.]],

         [[ 4.,  5.],
          [12., 13.]],

         [[ 6.,  7.],
          [14., 15.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, w4):
    """Return shape (b, w4*h, w, c) via 'b h (w w4) c -> b (w4 h) w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 8, 2), 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(403)


In [ ]:
#@title 💡 Solution — Problem 403
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, w4):
    return einops.rearrange(arr, 'b h (w w4) c -> b (w4 h) w c', w4=w4)


print(solve(t.arange(16.0).reshape(1, 1, 8, 2), 4))


#### Common mistakes

- **"Patch extraction needs sliding-window machinery."** — NON-overlapping
  patches are a pure reshape (split + merge); no windows, no copies of
  copies. Sliding (overlapping) windows are `x.unfold(...)`'s job — different
  task, check the word "non-overlapping".
- **"Space-to-depth loses spatial information."** — It's a bijection: every
  pixel gets a unique (channel, position) address, and depth-to-space
  inverts it exactly. What changes is which axis "sees" the detail.
- **"The channel packing order after s2d doesn't matter."** — Downstream
  code (or the grader) reads channels by index; (c p q) vs (p q c) are
  different tensors with equal shapes. The paren order IS the file format —
  get it from the task, verify with a position-valued example.
